### 📈 How Volatility Drag Impacts Wealth Creation

##### ▶️ Related Quant Guild Videos:

- [The 5 Papers That Built Modern Quant Finance](https://youtu.be/ZwS1gMGegrM)

- [I Bet You've Never Found Alpha (and I Can Prove It)](https://youtu.be/UzTJHs3-eT0)

- [Quant Ranks Retail Trading Mistakes that Blow Up Your Account](https://youtu.be/1mpNxBaBeOw)

- [Non-Stationarity and Why Market Timing Fails](https://youtu.be/7nvjrgqKjJE)

- [Quant Busts 3 Trading Myths with Math](https://youtu.be/wJfIk3VnubE)

- [How to Read Options Chains](https://youtu.be/RrRbz6oXwxE)

###### ______________________________________________________________________________________________________________________________________

##### [🚀 Master your Quantitative Skills with Quant Guild](https://quantguild.com)

##### [🛡️ Learn to Run a Personal Hedge Fund](https://quantguild.com/personal-hedge-fund)

##### [📚 Visit the Quant Guild Library for more Jupyter Notebooks](https://github.com/romanmichaelpaolucci/Quant-Guild-Library)

##### [📈 Interactive Brokers for Algorithmic Trading](https://www.interactivebrokers.com/mkt/?src=quantguildY&url=%2Fen%2Fwhyib%2Foverview.php)

##### [👾 Join the Quant Guild Discord Server](discord.com/invite/MJ4FU2c6c3)

---

##### 🐉 What is Volatility Drag

Volatility drag is a consequence of the geometric compounding of returns 

 
  $$
  P_n = P_0 \cdot \prod_{i=1}^n (1 + r_i) \implies 
  \log \frac{P_n}{P_0} = \sum_{i=1}^n \log(1+r_i) 
  $$
  
  For small $r$, $\log(1+r) \approx r - \frac{1}{2} r^2$,
  so the average log return (long-run growth rate) is approximately:
  $$
  \mathbb{E}[\log(1+r)] \approx \mathbb{E}[r] - \frac{1}{2}\mathrm{Var}(r)
  $$
  In words: **Mean compounded return $\approx$ arithmetic mean return minus volatility drag.**

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# Config
# ============================================================

SEED = 5
rng = np.random.default_rng(SEED)

YEARS = 15
STEPS_PER_YEAR = 12              # monthly steps
N_STEPS = YEARS * STEPS_PER_YEAR
DT = 1 / STEPS_PER_YEAR

INITIAL_VALUE = 1.0
START_DATE = "2025-01-01"
N_PATHS = 30

MU = 0.10                       # 30% annual arithmetic drift
SIGMA = 0.25                     # 22% annual volatility

FRAME_STRIDE = 3                 # reveal quarterly
FRAME_DURATION = 45
INITIAL_I = 3

# ============================================================
# Simulation and calculation helpers
# ============================================================

def simulate_gbm_path(s0, mu, sigma, n_steps, rng):
    z = rng.normal(size=n_steps)
    log_returns = (
        (mu - 0.5 * sigma**2) * DT
        + sigma * np.sqrt(DT) * z
    )
    return s0 * np.exp(np.r_[0.0, np.cumsum(log_returns)])

def gbm_theoretical_arithmetic_mean(s0, mu, times):
    return s0 * np.exp(mu * times)

def gbm_theoretical_geometric_mean(s0, mu, sigma, times):
    return s0 * np.exp((mu - 0.5 * sigma**2) * times)

def padded_range(values, pad_fraction=0.08, min_pad=0.10):
    values = np.asarray(values, dtype=float)
    v_min = float(np.nanmin(values))
    v_max = float(np.nanmax(values))
    if np.isclose(v_min, v_max):
        pad = max(abs(v_max) * pad_fraction, min_pad)
    else:
        pad = max((v_max - v_min) * pad_fraction, min_pad)
    return [v_min - pad, v_max + pad]

def path_proportions(path_values, arithmetic_mean, geometric_mean):
    above_arithmetic = 100.0 * np.mean(path_values > arithmetic_mean)
    below_arithmetic = 100.0 - above_arithmetic
    above_geometric = 100.0 * np.mean(path_values > geometric_mean)
    below_geometric = 100.0 - above_geometric
    return {
        "above": [above_arithmetic, above_geometric],
        "below": [below_arithmetic, below_geometric],
    }

def endpoint_text(label, n_points):
    if n_points <= 0:
        return []
    return [""] * (n_points - 1) + [label]

# ============================================================
# Time axis and GBM simulation
# ============================================================

dates = pd.date_range(
    start=START_DATE,
    periods=N_STEPS + 1,
    freq="MS",
)

times = np.arange(N_STEPS + 1) * DT

gbm_paths = np.column_stack([
    simulate_gbm_path(
        s0=INITIAL_VALUE,
        mu=MU,
        sigma=SIGMA,
        n_steps=N_STEPS,
        rng=rng,
    )
    for _ in range(N_PATHS)
])

gbm_arithmetic_mean = gbm_theoretical_arithmetic_mean(
    INITIAL_VALUE,
    MU,
    times,
)

gbm_geometric_mean = gbm_theoretical_geometric_mean(
    INITIAL_VALUE,
    MU,
    SIGMA,
    times,
)

# ============================================================
# Return decomposition
# ============================================================

arithmetic_return_pct = 100.0 * MU
volatility_drag_pct = 100.0 * 0.5 * SIGMA**2
geometric_return_pct = arithmetic_return_pct - volatility_drag_pct

if geometric_return_pct < 0:
    raise ValueError(
        "This stacked-bar design assumes MU - 0.5*SIGMA**2 is non-negative. "
        "Increase MU or reduce SIGMA for this visualization."
    )

# ============================================================
# Styling
# ============================================================

off_white = "#e0e0e0"
muted_white = "#b8b8b8"

path_color_above_geom = "#18d618"     # green
path_color_below_geom = "#ff3030"     # red

arithmetic_mean_color = "#00ff88"     # bright green
geometric_mean_color = "#ffd84d"      # yellow
mean_gap_fill = "rgba(255,216,77,0.10)"

above_color = "#18d618"
below_color = "#ff3030"

geometric_return_color = "#18d618"
volatility_drag_color = "#ff3030"

baseline_color = "#777777"

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.10)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white),
)

# ============================================================
# Figure layout
# ============================================================

fig = make_subplots(
    rows=2,
    cols=2,
    specs=[
        [{"type": "xy"}, {"type": "xy"}],
        [{"type": "xy", "colspan": 2}, None],
    ],
    row_heights=[0.72, 0.28],
    column_widths=[0.68, 0.32],
    horizontal_spacing=0.09,
    vertical_spacing=0.18,
    subplot_titles=(
        "GBM Paths: Convexity and Geometric Returns",
        "Share of Paths Above / Below Each Mean",
        "Volatility Drag: Arithmetic Return Decomposition",
    ),
)

initial_end = min(INITIAL_I, N_STEPS)
initial_n_points = initial_end + 1

# ============================================================
# Top-left panel: animated GBM paths
# ============================================================

for j in range(N_PATHS):
    y_values = gbm_paths[:initial_n_points, j]
    current_geometric_mean = gbm_geometric_mean[initial_end]

    path_color = (
        path_color_above_geom
        if y_values[-1] > current_geometric_mean
        else path_color_below_geom
    )

    fig.add_trace(
        go.Scatter(
            x=dates[:initial_n_points],
            y=y_values,
            mode="lines",
            line=dict(color=path_color, width=1.5),
            opacity=0.50,
            name="GBM simulations" if j == 0 else f"GBM path {j + 1}",
            legendgroup="gbm-paths",
            showlegend=False,
            hovertemplate=(
                f"GBM path {j + 1}<br>"
                "Date: %{x|%Y-%m-%d}<br>"
                "Value: %{y:.4f}<extra></extra>"
            ),
        ),
        row=1,
        col=1,
    )

fig.add_trace(
    go.Scatter(
        x=dates[:initial_n_points],
        y=gbm_arithmetic_mean[:initial_n_points],
        mode="lines+text",
        line=dict(color=arithmetic_mean_color, width=4),
        text=endpoint_text("  Arithmetic mean", initial_n_points),
        textposition="top right",
        textfont=dict(color=arithmetic_mean_color, size=11),
        name="Theoretical arithmetic mean",
        legendgroup="arithmetic-mean",
        showlegend=False,
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "E[Sₜ]: %{y:.4f}<extra></extra>"
        ),
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=dates[:initial_n_points],
        y=gbm_geometric_mean[:initial_n_points],
        mode="lines+text",
        line=dict(color=geometric_mean_color, width=4, dash="dash"),
        fill="tonexty",
        fillcolor=mean_gap_fill,
        text=endpoint_text("  Geometric mean", initial_n_points),
        textposition="bottom right",
        textfont=dict(color=geometric_mean_color, size=11),
        name="Theoretical geometric mean",
        legendgroup="geometric-mean",
        showlegend=False,
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "exp(E[log Sₜ]): %{y:.4f}<extra></extra>"
        ),
    ),
    row=1,
    col=1,
)

fig.add_hline(
    y=INITIAL_VALUE,
    line=dict(color=baseline_color, width=1, dash="dash"),
    opacity=0.65,
    row=1,
    col=1,
)

# ============================================================
# Top-right panel: proportions above/below each mean
# ============================================================

initial_proportions = path_proportions(
    gbm_paths[initial_end, :],
    gbm_arithmetic_mean[initial_end],
    gbm_geometric_mean[initial_end],
)

above_x = [-0.18, 0.82]
below_x = [0.18, 1.18]
bar_width = [0.32, 0.32]

fig.add_trace(
    go.Bar(
        x=above_x,
        y=initial_proportions["above"],
        width=bar_width,
        marker=dict(color=above_color),
        name="Above mean",
        legendgroup="proportions-above",
        showlegend=False,
        text=[f"{value:.1f}%" for value in initial_proportions["above"]],
        textposition="inside",
        insidetextanchor="middle",
        hovertemplate=(
            "%{customdata}<br>"
            "Above: %{y:.1f}%<extra></extra>"
        ),
        customdata=["Arithmetic mean", "Geometric mean"],
    ),
    row=1,
    col=2,
)

fig.add_trace(
    go.Bar(
        x=below_x,
        y=initial_proportions["below"],
        width=bar_width,
        marker=dict(color=below_color),
        name="Below mean",
        legendgroup="proportions-below",
        showlegend=False,
        text=[f"{value:.1f}%" for value in initial_proportions["below"]],
        textposition="inside",
        insidetextanchor="middle",
        hovertemplate=(
            "%{customdata}<br>"
            "Below: %{y:.1f}%<extra></extra>"
        ),
        customdata=["Arithmetic mean", "Geometric mean"],
    ),
    row=1,
    col=2,
)

# ============================================================
# Bottom panel: arithmetic return eaten by volatility drag
# 
# Fix for forced redraw:
# Place bottom panel traces in *every animation frame*
# so they never flicker/disappear due to a full redraw.
# 

bottom_bar_geometric = go.Bar(
    x=[geometric_return_pct],
    y=["Arithmetic return"],
    base=[0.0],
    orientation="h",
    width=0.52,
    marker=dict(color=geometric_return_color),
    name="Geometric return retained",
    legendgroup="geometric-return-retained",
    showlegend=False,
    text=[f"Geometric return retained: {geometric_return_pct:.2f}%"],
    textposition="inside",
    insidetextanchor="middle",
    hovertemplate=(
        "Geometric return retained<br>"
        f"{geometric_return_pct:.2f}%<extra></extra>"
    ),
)
bottom_bar_drag = go.Bar(
    x=[volatility_drag_pct],
    y=["Arithmetic return"],
    base=[geometric_return_pct],
    orientation="h",
    width=0.52,
    marker=dict(color=volatility_drag_color),
    name="Volatility drag",
    legendgroup="volatility-drag",
    showlegend=False,
    text=[f"Drag: {volatility_drag_pct:.2f}%"],
    textposition="inside",
    insidetextanchor="middle",
    hovertemplate=(
        "Volatility drag = ½σ²<br>"
        f"{volatility_drag_pct:.2f}%<extra></extra>"
    ),
)

fig.add_trace(bottom_bar_geometric, row=2, col=1)
fig.add_trace(bottom_bar_drag, row=2, col=1)

fig.add_vline(
    x=arithmetic_return_pct,
    line=dict(color=off_white, width=2, dash="dot"),
    opacity=0.85,
    row=2,
    col=1,
)

fig.add_annotation(
    x=arithmetic_return_pct,
    y="Arithmetic return",
    text=f"Arithmetic return μ = {arithmetic_return_pct:.2f}%",
    showarrow=True,
    arrowhead=2,
    ax=0,
    ay=-42,
    font=dict(color=off_white, size=12),
    arrowcolor=off_white,
    bgcolor="rgba(30,30,30,0.75)",
    bordercolor="rgba(255,255,255,0.25)",
    borderwidth=1,
    row=2,
    col=1,
)

# ============================================================
# Animation frames
# ============================================================

frames = []
slider_steps = []

frame_indices = list(range(initial_end, N_STEPS + 1, FRAME_STRIDE))
if frame_indices[-1] != N_STEPS:
    frame_indices.append(N_STEPS)

# Animated trace indices for all traces (top and bottom) in plot order:
# [gbm paths][arithmetic mean][geometric mean][above bar][below bar][bottom bar geometric][bottom bar drag]
n_top_traces = N_PATHS + 4
bottom_bar_geometric_idx = n_top_traces
bottom_bar_drag_idx = n_top_traces + 1

for i in frame_indices:
    frame_name = f"f{i}"
    n_points = i + 1
    frame_data = []

    current_geometric_mean = gbm_geometric_mean[i]

    # Animated GBM paths.
    for j in range(N_PATHS):
        y_values = gbm_paths[:n_points, j]
        path_color = (
            path_color_above_geom
            if y_values[-1] > current_geometric_mean
            else path_color_below_geom
        )
        frame_data.append(
            go.Scatter(
                x=dates[:n_points],
                y=y_values,
                mode="lines",
                opacity=0.50,
                line=dict(color=path_color, width=1.5),
                showlegend=False,
            )
        )

    # Theoretical arithmetic mean.
    frame_data.append(
        go.Scatter(
            x=dates[:n_points],
            y=gbm_arithmetic_mean[:n_points],
            mode="lines+text",
            line=dict(color=arithmetic_mean_color, width=4),
            text=endpoint_text("  Arithmetic mean", n_points),
            textposition="top right",
            textfont=dict(color=arithmetic_mean_color, size=11),
            showlegend=False,
        )
    )

    # Theoretical geometric mean and shaded mean gap.
    frame_data.append(
        go.Scatter(
            x=dates[:n_points],
            y=gbm_geometric_mean[:n_points],
            mode="lines+text",
            line=dict(color=geometric_mean_color, width=4, dash="dash"),
            fill="tonexty",
            fillcolor=mean_gap_fill,
            text=endpoint_text("  Geometric mean", n_points),
            textposition="bottom right",
            textfont=dict(color=geometric_mean_color, size=11),
            showlegend=False,
        )
    )

    # Proportions at the current animation date.
    proportions = path_proportions(
        gbm_paths[i, :],
        gbm_arithmetic_mean[i],
        gbm_geometric_mean[i],
    )

    frame_data.append(
        go.Bar(
            x=above_x,
            y=proportions["above"],
            width=bar_width,
            marker=dict(color=above_color),
            text=[f"{value:.1f}%" for value in proportions["above"]],
            textposition="inside",
            insidetextanchor="middle",
            customdata=["Arithmetic mean", "Geometric mean"],
            showlegend=False,
        )
    )

    frame_data.append(
        go.Bar(
            x=below_x,
            y=proportions["below"],
            width=bar_width,
            marker=dict(color=below_color),
            text=[f"{value:.1f}%" for value in proportions["below"]],
            textposition="inside",
            insidetextanchor="middle",
            customdata=["Arithmetic mean", "Geometric mean"],
            showlegend=False,
        )
    )

    # --- FIX: Add bottom panel traces to every frame ---
    frame_data.append(bottom_bar_geometric)
    frame_data.append(bottom_bar_drag)
    frame_traces = list(range(n_top_traces + 2))

    frames.append(
        go.Frame(
            data=frame_data,
            traces=frame_traces,
            name=frame_name,
        )
    )

    elapsed_years = i / STEPS_PER_YEAR
    slider_steps.append({
        "args": [
            [frame_name],
            {
                "frame": {"duration": 0, "redraw": True},
                "mode": "immediate",
                "fromcurrent": True,
            },
        ],
        "label": f"{elapsed_years:.1f}Y",
        "method": "animate",
    })

fig.frames = frames

# ============================================================
# Axis ranges
# ============================================================

gbm_y_values = np.r_[
    gbm_paths.ravel(),
    gbm_arithmetic_mean,
    gbm_geometric_mean,
]

gbm_y_range = padded_range(
    gbm_y_values,
    pad_fraction=0.08,
    min_pad=0.20,
)

return_x_max = max(arithmetic_return_pct * 1.16, arithmetic_return_pct + 2.0)

# ============================================================
# Layout
# ============================================================

fig.update_layout(
    title=dict(
        text=(
            "GBM Convexity, Mean Location, and Volatility Drag"
            f"<br><sup>μ = {MU:.1%}, σ = {SIGMA:.1%}, "
            f"geometric rate = μ - ½σ² = {geometric_return_pct:.2f}%</sup>"
        ),
        x=0.5,
        font=dict(color=off_white),
    ),
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=800,
    width=1200,
    margin=dict(t=125, b=130, r=55, l=85),
    barmode="overlay",
    bargap=0.18,
    hovermode="closest",
    showlegend=False,
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [
                    None,
                    {
                        "frame": {
                            "duration": FRAME_DURATION,
                            "redraw": True,
                        },
                        "transition": {"duration": 0},
                        "fromcurrent": True,
                    },
                ],
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [
                    [None],
                    {
                        "frame": {"duration": 0, "redraw": True},
                        "mode": "immediate",
                        "fromcurrent": True,
                    },
                ],
            },
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 70},
        "showactive": False,
        "x": 0.10,
        "xanchor": "right",
        "y": -0.03,
        "yanchor": "top",
    }],
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {"visible": False},
        "transition": {"duration": 0},
        "pad": {"b": 10, "t": 40},
        "len": 0.85,
        "x": 0.15,
        "y": -0.03,
        "steps": slider_steps,
    }],
)

fig.update_annotations(font=dict(color=off_white, size=16))

fig.update_xaxes(
    axis_style,
    row=1,
    col=1,
    range=[dates[0], dates[-1]],
    title_text="Date",
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=1,
    range=gbm_y_range,
    title_text="Value",
)

fig.update_xaxes(
    axis_style,
    row=1,
    col=2,
    range=[-0.55, 1.55],
    tickmode="array",
    tickvals=[0.0, 1.0],
    ticktext=["Arithmetic<br>mean", "Geometric<br>mean"],
    title_text="Reference level",
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=2,
    range=[0, 105],
    dtick=20,
    ticksuffix="%",
    title_text="Proportion of paths",
)

fig.update_xaxes(
    axis_style,
    row=2,
    col=1,
    range=[0, return_x_max],
    ticksuffix="%",
    title_text="Annual return",
)

fig.update_yaxes(
    axis_style,
    row=2,
    col=1,
    showgrid=False,
    title_text="",
)

# ============================================================
# Display only — no HTML export
# ============================================================

fig.show()

###### ______________________________________________________________________________________________________________________________________

##### 📉 Impact of Volatility Drag on Long-Run Returns

Volatility drag can erode the long-run expected returns of your portfolio

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots


# ============================================================
# Config
# ============================================================

SEED = 7

STEPS_PER_YEAR = 12                  # monthly GBM simulation steps
DT = 1.0 / STEPS_PER_YEAR

INITIAL_VALUE = 1.0
START_DATE = "2025-01-01"

N_DISPLAY_PATHS = 30
FRAME_STRIDE = 6                     # reveal one frame every six months
FRAME_DURATION = 70
INITIAL_I = 3                        # first frame at 0.25 years

HORIZON_OPTIONS = [5, 10, 15, 20, 25, 30]
DEFAULT_HORIZON = 30

# High-volatility case:
#
#     geometric return = mu - 0.5 * sigma^2
#                      = 18% - 16%
#                      = 2%.
#
# The arithmetic return looks attractive, but almost all of it is consumed
# by volatility drag.
HIGH_VOL_MU = 0.18
HIGH_VOL_GEOMETRIC = 0.02
HIGH_VOL_SIGMA = np.sqrt(
    2.0 * (HIGH_VOL_MU - HIGH_VOL_GEOMETRIC)
)

SCENARIO_INPUTS = [
    {
        "key": "low",
        "title": "Lower arithmetic return / lower volatility",
        "mu": 0.08,
        "sigma": 0.15,
    },
    {
        "key": "high",
        "title": "Higher arithmetic return / excessive volatility",
        "mu": HIGH_VOL_MU,
        "sigma": HIGH_VOL_SIGMA,
    },
]


# ============================================================
# Styling — follows the attached transparent Plotly-dark style
# ============================================================

TRANSPARENT = "rgba(0,0,0,0)"

OFF_WHITE = "#e0e0e0"
MUTED_WHITE = "#b8b8b8"

PATH_COLOR_ABOVE_GEOMETRIC = "#18d618"
PATH_COLOR_BELOW_GEOMETRIC = "#ff3030"

ARITHMETIC_MEAN_COLOR = "#00ff88"
GEOMETRIC_MEAN_COLOR = "#ffd84d"
MEAN_GAP_FILL = "rgba(255,216,77,0.10)"

GEOMETRIC_RETURN_COLOR = "#18d618"
VOLATILITY_DRAG_COLOR = "#ff3030"

BASELINE_COLOR = "#777777"

AXIS_STYLE = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.10)",
    tickfont=dict(color=OFF_WHITE),
    linecolor=OFF_WHITE,
    zeroline=False,
    title_font=dict(color=OFF_WHITE),
)


# ============================================================
# Simulation helpers
# ============================================================

def simulate_gbm_paths(s0, mu, sigma, shocks):
    """
    Simulate geometric Brownian motion paths from supplied standard-normal
    shocks.

        dS_t / S_t = mu dt + sigma dW_t

        S_t = S_0 exp[(mu - 0.5 sigma^2)t + sigma W_t]
    """
    log_increments = (
        (mu - 0.5 * sigma**2) * DT
        + sigma * np.sqrt(DT) * shocks
    )

    cumulative_log_returns = np.vstack(
        [
            np.zeros((1, shocks.shape[1])),
            np.cumsum(log_increments, axis=0),
        ]
    )

    return s0 * np.exp(cumulative_log_returns)


def arithmetic_wealth_path(s0, mu, times):
    """Expected GBM wealth: E[S_t] = S_0 exp(mu t)."""
    return s0 * np.exp(mu * times)


def geometric_wealth_path(s0, mu, sigma, times):
    """
    Typical/median GBM wealth:

        exp(E[log S_t]) = S_0 exp[(mu - 0.5 sigma^2)t]
    """
    return s0 * np.exp(
        (mu - 0.5 * sigma**2) * times
    )


def padded_log10_range(values, pad_decades=0.10):
    """
    Return a padded Plotly logarithmic-axis range.

    Plotly expects log-axis ranges in base-10 logarithmic coordinates.
    """
    values = np.asarray(values, dtype=float)
    values = values[
        np.isfinite(values)
        & (values > 0.0)
    ]

    if values.size == 0:
        raise ValueError(
            "A logarithmic wealth axis requires positive values."
        )

    return [
        np.log10(values.min()) - pad_decades,
        np.log10(values.max()) + pad_decades,
    ]


def endpoint_text(label, n_points):
    """Place a direct label only at the newest point of an animated line."""
    if n_points <= 0:
        return []

    return [""] * (n_points - 1) + [label]


def copy_trace(trace, **updates):
    """Return a new Plotly trace copied from an existing trace."""
    trace_json = trace.to_plotly_json()
    trace_json.update(updates)

    if trace_json.get("type") == "bar":
        return go.Bar(trace_json)

    return go.Scatter(trace_json)


# ============================================================
# Horizon data
# ============================================================

def build_horizon_data(years):
    """
    Generate an independent deterministic simulation for one horizon.

    Plotly's native browser controls cannot execute Python again. Therefore,
    each selectable horizon is generated in advance and stored in the figure.
    """
    n_steps = years * STEPS_PER_YEAR

    # A different deterministic seed gives every horizon a freshly generated
    # simulation while still making repeated script runs reproducible.
    rng = np.random.default_rng(
        SEED + 1009 * years
    )

    dates = pd.date_range(
        start=START_DATE,
        periods=n_steps + 1,
        freq="MS",
    )

    times = np.arange(n_steps + 1) * DT

    # Both scenarios use the same standardized shocks within a horizon.
    # This makes their visual difference primarily a function of mu and sigma.
    common_shocks = rng.normal(
        size=(n_steps, N_DISPLAY_PATHS)
    )

    scenarios = []

    for source in SCENARIO_INPUTS:
        scenario = dict(source)

        scenario["drag"] = (
            0.5 * scenario["sigma"]**2
        )
        scenario["geometric"] = (
            scenario["mu"] - scenario["drag"]
        )

        if scenario["geometric"] < 0.0:
            raise ValueError(
                "This retained/drag display requires a non-negative "
                "geometric return."
            )

        scenario["paths"] = simulate_gbm_paths(
            s0=INITIAL_VALUE,
            mu=scenario["mu"],
            sigma=scenario["sigma"],
            shocks=common_shocks,
        )

        scenario["arithmetic_wealth"] = arithmetic_wealth_path(
            s0=INITIAL_VALUE,
            mu=scenario["mu"],
            times=times,
        )

        scenario["geometric_wealth"] = geometric_wealth_path(
            s0=INITIAL_VALUE,
            mu=scenario["mu"],
            sigma=scenario["sigma"],
            times=times,
        )

        scenario["y_range"] = padded_log10_range(
            np.r_[
                scenario["paths"].ravel(),
                scenario["arithmetic_wealth"],
                scenario["geometric_wealth"],
                INITIAL_VALUE,
            ]
        )

        scenario["arithmetic_pct"] = (
            100.0 * scenario["mu"]
        )
        scenario["drag_pct"] = (
            100.0 * scenario["drag"]
        )
        scenario["geometric_pct"] = (
            100.0 * scenario["geometric"]
        )

        scenarios.append(scenario)

    initial_step = min(INITIAL_I, n_steps)

    frame_indices = list(
        range(
            initial_step,
            n_steps + 1,
            FRAME_STRIDE,
        )
    )

    if frame_indices[-1] != n_steps:
        frame_indices.append(n_steps)

    return {
        "years": years,
        "n_steps": n_steps,
        "dates": dates,
        "times": times,
        "scenarios": scenarios,
        "initial_step": initial_step,
        "frame_indices": frame_indices,
    }


# ============================================================
# Dynamic labels and controls
# ============================================================

def figure_title(step, horizon_data):
    """Title text for one animation step."""
    elapsed_years = step / STEPS_PER_YEAR
    low_scenario, high_scenario = horizon_data["scenarios"]

    low_typical = (
        low_scenario["geometric_wealth"][step]
    )
    high_typical = (
        high_scenario["geometric_wealth"][step]
    )

    return (
        "Volatility drag: arithmetic return is not geometric compounding"
        "<br><sup>"
        f"Selected horizon: {horizon_data['years']} years"
        f" &nbsp;|&nbsp; Elapsed: {elapsed_years:.1f} years"
        f" &nbsp;|&nbsp; Low-vol typical wealth: {low_typical:.2f}×"
        f" &nbsp;|&nbsp; High-vol typical wealth: {high_typical:.2f}×"
        "</sup>"
    )


def play_pause_menu(frame_names):
    """
    Build a Plotly-native replay/pause menu for exactly one horizon.

    fromcurrent=False makes Replay start from the beginning every time.
    """
    return {
        "type": "buttons",
        "buttons": [
            {
                "label": "▶ Replay selected horizon",
                "method": "animate",
                "args": [
                    frame_names,
                    {
                        "frame": {
                            "duration": FRAME_DURATION,
                            "redraw": True,
                        },
                        "transition": {
                            "duration": 0
                        },
                        "mode": "immediate",
                        "fromcurrent": False,
                    },
                ],
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [
                    [None],
                    {
                        "frame": {
                            "duration": 0,
                            "redraw": True,
                        },
                        "transition": {
                            "duration": 0
                        },
                        "mode": "immediate",
                        "fromcurrent": True,
                    },
                ],
            },
        ],
        "direction": "left",
        "pad": {
            "r": 10,
            "t": 10,
        },
        "showactive": False,
        "x": 0.01,
        "xanchor": "left",
        "y": -0.14,
        "yanchor": "top",
        "bgcolor": TRANSPARENT,
        "bordercolor": "rgba(255,255,255,0.22)",
        "borderwidth": 1,
        "font": {
            "color": OFF_WHITE
        },
    }


# ============================================================
# Trace builders
# ============================================================

def make_path_trace(
    dates,
    path_values,
    path_index,
    current_geometric_wealth,
    years,
    visible=True,
):
    """Build one animated simulated-path trace."""
    path_color = (
        PATH_COLOR_ABOVE_GEOMETRIC
        if path_values[-1] > current_geometric_wealth
        else PATH_COLOR_BELOW_GEOMETRIC
    )

    return go.Scatter(
        x=dates,
        y=path_values,
        mode="lines",
        line=dict(
            color=path_color,
            width=1.5,
        ),
        opacity=0.50,
        visible=visible,
        showlegend=False,
        hovertemplate=(
            f"{years}Y path {path_index + 1}<br>"
            "Date: %{x|%Y-%m-%d}<br>"
            "Wealth: %{y:.4f}×"
            "<extra></extra>"
        ),
    )


def make_arithmetic_mean_trace(
    dates,
    wealth_values,
    visible=True,
):
    """Build the arithmetic-mean line with a direct endpoint label."""
    n_points = len(wealth_values)

    return go.Scatter(
        x=dates,
        y=wealth_values,
        mode="lines+text",
        line=dict(
            color=ARITHMETIC_MEAN_COLOR,
            width=4,
        ),
        text=endpoint_text(
            "  Arithmetic mean",
            n_points,
        ),
        textposition="top right",
        textfont=dict(
            color=ARITHMETIC_MEAN_COLOR,
            size=11,
        ),
        cliponaxis=False,
        visible=visible,
        showlegend=False,
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "Arithmetic expectation: %{y:.4f}×"
            "<extra></extra>"
        ),
    )


def make_geometric_mean_trace(
    dates,
    wealth_values,
    visible=True,
):
    """
    Build the geometric-mean line.

    It follows the arithmetic trace in plot order, so fill='tonexty' shades
    the volatility-drag gap between the two mean lines.
    """
    n_points = len(wealth_values)

    return go.Scatter(
        x=dates,
        y=wealth_values,
        mode="lines+text",
        line=dict(
            color=GEOMETRIC_MEAN_COLOR,
            width=4,
            dash="dash",
        ),
        fill="tonexty",
        fillcolor=MEAN_GAP_FILL,
        text=endpoint_text(
            "  Geometric mean",
            n_points,
        ),
        textposition="bottom right",
        textfont=dict(
            color=GEOMETRIC_MEAN_COLOR,
            size=11,
        ),
        cliponaxis=False,
        visible=visible,
        showlegend=False,
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "Typical wealth: %{y:.4f}×"
            "<extra></extra>"
        ),
    )


def make_bottom_bar_traces(scenarios):
    """
    Build the four fixed lower-panel traces.

    These same bar traces are inserted into every animation frame. That is
    intentional: Plotly frames use redraw=True, so including the bars prevents
    them from flickering or disappearing during animation.

    This version removes text overlays on the bars as requested.
    """
    traces = []

    for scenario in scenarios:
        traces.append(
            go.Bar(
                x=[scenario["geometric_pct"]],
                y=["Arithmetic return"],
                base=[0.0],
                orientation="h",
                width=0.52,
                marker=dict(
                    color=GEOMETRIC_RETURN_COLOR
                ),
                # Remove text, textposition, insidetextanchor, textfont
                showlegend=False,
                hovertemplate=(
                    "Geometric return retained<br>"
                    f"{scenario['geometric_pct']:.2f}%"
                    "<extra></extra>"
                ),
            )
        )

        traces.append(
            go.Bar(
                x=[scenario["drag_pct"]],
                y=["Arithmetic return"],
                base=[scenario["geometric_pct"]],
                orientation="h",
                width=0.52,
                marker=dict(
                    color=VOLATILITY_DRAG_COLOR
                ),
                # Remove text, textposition, insidetextanchor, textfont
                showlegend=False,
                hovertemplate=(
                    "Volatility drag = ½σ²<br>"
                    f"{scenario['drag_pct']:.2f}%"
                    "<extra></extra>"
                ),
            )
        )

    return traces


def make_initial_top_traces(horizon_data, visible):
    """Build all upper-panel traces at a horizon's reset position."""
    step = horizon_data["initial_step"]
    n_points = step + 1
    dates = horizon_data["dates"][:n_points]

    traces = []

    for scenario in horizon_data["scenarios"]:
        current_geometric = (
            scenario["geometric_wealth"][step]
        )

        for path_index in range(N_DISPLAY_PATHS):
            traces.append(
                make_path_trace(
                    dates=dates,
                    path_values=scenario["paths"][
                        :n_points,
                        path_index,
                    ],
                    path_index=path_index,
                    current_geometric_wealth=current_geometric,
                    years=horizon_data["years"],
                    visible=visible,
                )
            )

        traces.append(
            make_arithmetic_mean_trace(
                dates=dates,
                wealth_values=scenario[
                    "arithmetic_wealth"
                ][:n_points],
                visible=visible,
            )
        )

        traces.append(
            make_geometric_mean_trace(
                dates=dates,
                wealth_values=scenario[
                    "geometric_wealth"
                ][:n_points],
                visible=visible,
            )
        )

    return traces


def make_animation_top_traces(horizon_data, step):
    """Build the upper trace updates for one animation frame."""
    n_points = step + 1
    dates = horizon_data["dates"][:n_points]

    traces = []

    for scenario in horizon_data["scenarios"]:
        current_geometric = (
            scenario["geometric_wealth"][step]
        )

        for path_index in range(N_DISPLAY_PATHS):
            traces.append(
                make_path_trace(
                    dates=dates,
                    path_values=scenario["paths"][
                        :n_points,
                        path_index,
                    ],
                    path_index=path_index,
                    current_geometric_wealth=current_geometric,
                    years=horizon_data["years"],
                    visible=True,
                )
            )

        traces.append(
            make_arithmetic_mean_trace(
                dates=dates,
                wealth_values=scenario[
                    "arithmetic_wealth"
                ][:n_points],
                visible=True,
            )
        )

        traces.append(
            make_geometric_mean_trace(
                dates=dates,
                wealth_values=scenario[
                    "geometric_wealth"
                ][:n_points],
                visible=True,
            )
        )

    return traces


# ============================================================
# Figure builder
# ============================================================

def build_figure(default_horizon=DEFAULT_HORIZON):
    """
    Build one self-contained Plotly figure with:

    - two GBM path panels;
    - two fixed retained/drag decomposition panels;
    - a native horizon slider for 5Y through 30Y;
    - a replay/pause button;
    - no legend;
    - transparent paper and plot backgrounds.
    """
    if default_horizon not in HORIZON_OPTIONS:
        raise ValueError(
            "default_horizon must be one of "
            f"{HORIZON_OPTIONS}."
        )

    horizon_store = {
        years: build_horizon_data(years)
        for years in HORIZON_OPTIONS
    }

    base_scenarios = horizon_store[
        default_horizon
    ]["scenarios"]

    subplot_titles = []

    for scenario in base_scenarios:
        subplot_titles.append(
            f"{scenario['title']}"
            "<br><sup>"
            f"μ = {scenario['mu']:.1%}, "
            f"σ = {scenario['sigma']:.1%}, "
            f"geometric = {scenario['geometric']:.1%}"
            "</sup>"
        )

    for scenario in base_scenarios:
        subplot_titles.append(
            "Volatility drag: arithmetic-return decomposition"
            "<br><sup>"
            f"{scenario['arithmetic_pct']:.2f}% arithmetic = "
            f"{scenario['geometric_pct']:.2f}% retained + "
            f"{scenario['drag_pct']:.2f}% drag"
            "</sup>"
        )

    fig = make_subplots(
        rows=2,
        cols=2,
        column_widths=[
            0.50,
            0.50,
        ],
        row_heights=[
            0.74,
            0.26,
        ],
        horizontal_spacing=0.08,
        vertical_spacing=0.17,
        subplot_titles=tuple(
            subplot_titles
        ),
    )

    # Each horizon has its own group of upper traces.
    horizon_trace_indices = {}
    horizon_reset_traces = {}

    for years in HORIZON_OPTIONS:
        horizon_data = horizon_store[years]
        is_default = (
            years == default_horizon
        )

        initial_traces = make_initial_top_traces(
            horizon_data=horizon_data,
            visible=is_default,
        )

        horizon_reset_traces[years] = [
            copy_trace(
                trace,
                visible=True,
            )
            for trace in initial_traces
        ]

        indices = []

        # Each scenario contributes N paths, an arithmetic line, and a
        # geometric line. The first scenario is placed in column one and the
        # second in column two.
        traces_per_scenario = (
            N_DISPLAY_PATHS + 2
        )

        for trace_number, trace in enumerate(
            initial_traces
        ):
            scenario_index = (
                trace_number
                // traces_per_scenario
            )
            col = scenario_index + 1

            fig.add_trace(
                trace,
                row=1,
                col=col,
            )

            indices.append(
                len(fig.data) - 1
            )

        horizon_trace_indices[years] = indices

    # Add one shared set of fixed lower bars.
    bottom_bar_traces = make_bottom_bar_traces(
        base_scenarios
    )

    bottom_trace_indices = []

    for trace_number, trace in enumerate(
        bottom_bar_traces
    ):
        col = (
            trace_number // 2
        ) + 1

        fig.add_trace(
            trace,
            row=2,
            col=col,
        )

        bottom_trace_indices.append(
            len(fig.data) - 1
        )

    # Arithmetic-return endpoint lines and callouts.
    for col, scenario in enumerate(
        base_scenarios,
        start=1,
    ):
        fig.add_vline(
            x=scenario["arithmetic_pct"],
            line=dict(
                color=OFF_WHITE,
                width=2,
                dash="dot",
            ),
            opacity=0.85,
            row=2,
            col=col,
        )

        fig.add_annotation(
            x=scenario["arithmetic_pct"],
            y="Arithmetic return",
            text=(
                "Arithmetic return μ = "
                f"{scenario['arithmetic_pct']:.2f}%"
            ),
            showarrow=True,
            arrowhead=2,
            ax=0,
            ay=-42,
            font=dict(
                color=OFF_WHITE,
                size=12,
            ),
            arrowcolor=OFF_WHITE,
            bgcolor="rgba(30,30,30,0.75)",
            bordercolor="rgba(255,255,255,0.25)",
            borderwidth=1,
            row=2,
            col=col,
        )

    # Initial-value baselines in both upper panels.
    for col in (1, 2):
        fig.add_hline(
            y=INITIAL_VALUE,
            line=dict(
                color=BASELINE_COLOR,
                width=1,
                dash="dash",
            ),
            opacity=0.65,
            row=1,
            col=col,
        )

    # ========================================================
    # Horizon animation frames
    # ========================================================

    frames = []
    frame_names_by_horizon = {}

    for years in HORIZON_OPTIONS:
        horizon_data = horizon_store[years]
        trace_indices = horizon_trace_indices[
            years
        ]
        frame_names = []

        for step in horizon_data[
            "frame_indices"
        ]:
            frame_name = (
                f"h{years:02d}_f{step:04d}"
            )
            frame_names.append(
                frame_name
            )

            frame_data = (
                make_animation_top_traces(
                    horizon_data=horizon_data,
                    step=step,
                )
                + [
                    copy_trace(
                        trace,
                        visible=True,
                    )
                    for trace in bottom_bar_traces
                ]
            )

            # Including the four lower bars in every frame is deliberate.
            # It preserves them when redraw=True and prevents flicker.
            frame_trace_indices = (
                trace_indices
                + bottom_trace_indices
            )

            frames.append(
                go.Frame(
                    data=frame_data,
                    traces=frame_trace_indices,
                    layout=go.Layout(
                        title=dict(
                            text=figure_title(
                                step,
                                horizon_data,
                            )
                        )
                    ),
                    name=frame_name,
                    group=f"horizon_{years}",
                )
            )

        frame_names_by_horizon[
            years
        ] = frame_names

    # ========================================================
    # Horizon-reset frames
    # ========================================================

    # A selector frame does three things:
    #
    # 1. hides every non-selected horizon;
    # 2. restores the selected horizon to its initial data;
    # 3. rewires Replay to the selected horizon's frame sequence.
    #
    # This means changing the native slider always gives a clean restart,
    # even after that horizon was previously animated to its endpoint.

    all_top_trace_indices = [
        index
        for years in HORIZON_OPTIONS
        for index in horizon_trace_indices[
            years
        ]
    ]

    selector_frame_names = {}

    for selected_years in HORIZON_OPTIONS:
        selected_data = horizon_store[
            selected_years
        ]
        selector_name = (
            f"select_h{selected_years:02d}"
        )
        selector_frame_names[
            selected_years
        ] = selector_name

        selector_data = []
        selector_indices = []

        selected_trace_map = dict(
            zip(
                horizon_trace_indices[
                    selected_years
                ],
                horizon_reset_traces[
                    selected_years
                ],
            )
        )

        for index in all_top_trace_indices:
            selector_indices.append(
                index
            )

            if index in selected_trace_map:
                selector_data.append(
                    copy_trace(
                        selected_trace_map[
                            index
                        ],
                        visible=True,
                    )
                )
            else:
                # All upper traces are Scatter traces.
                selector_data.append(
                    go.Scatter(
                        visible=False
                    )
                )

        # Keep all four lower bars visible and explicitly redraw them.
        for index, trace in zip(
            bottom_trace_indices,
            bottom_bar_traces,
        ):
            selector_indices.append(
                index
            )
            selector_data.append(
                copy_trace(
                    trace,
                    visible=True,
                )
            )

        frames.append(
            go.Frame(
                data=selector_data,
                traces=selector_indices,
                layout=go.Layout(
                    title=dict(
                        text=figure_title(
                            selected_data[
                                "initial_step"
                            ],
                            selected_data,
                        )
                    ),
                    xaxis=dict(
                        range=[
                            selected_data[
                                "dates"
                            ][0],
                            selected_data[
                                "dates"
                            ][-1],
                        ]
                    ),
                    xaxis2=dict(
                        range=[
                            selected_data[
                                "dates"
                            ][0],
                            selected_data[
                                "dates"
                            ][-1],
                        ]
                    ),
                    yaxis=dict(
                        range=selected_data[
                            "scenarios"
                        ][0]["y_range"]
                    ),
                    yaxis2=dict(
                        range=selected_data[
                            "scenarios"
                        ][1]["y_range"]
                    ),
                    updatemenus=[
                        play_pause_menu(
                            frame_names_by_horizon[
                                selected_years
                            ]
                        )
                    ],
                ),
                name=selector_name,
                group="horizon_selector",
            )
        )

    fig.frames = frames

    # ========================================================
    # Native horizon slider
    # ========================================================

    horizon_slider_steps = []

    for years in HORIZON_OPTIONS:
        horizon_slider_steps.append(
            {
                "label": f"{years}Y",
                "method": "animate",
                "args": [
                    [
                        selector_frame_names[
                            years
                        ]
                    ],
                    {
                        "frame": {
                            "duration": 0,
                            "redraw": True,
                        },
                        "transition": {
                            "duration": 0
                        },
                        "mode": "immediate",
                        "fromcurrent": False,
                    },
                ],
            }
        )

    default_data = horizon_store[
        default_horizon
    ]

    default_active = HORIZON_OPTIONS.index(
        default_horizon
    )

    return_x_max = (
        max(
            scenario["arithmetic_pct"]
            for scenario in base_scenarios
        )
        * 1.18
    )

    # ========================================================
    # Layout
    # ========================================================

    fig.update_layout(
        title=dict(
            text=figure_title(
                default_data[
                    "initial_step"
                ],
                default_data,
            ),
            x=0.5,
            font=dict(
                color=OFF_WHITE
            ),
        ),
        template="plotly_dark",

        # Explicit transparency, matching the attached code.
        paper_bgcolor=TRANSPARENT,
        plot_bgcolor=TRANSPARENT,

        font=dict(
            color=OFF_WHITE
        ),
        height=800,
        width=1200,
        margin=dict(
            t=125,
            b=165,
            r=55,
            l=85,
        ),
        barmode="overlay",
        bargap=0.18,
        hovermode="closest",
        showlegend=False,
        uniformtext_minsize=8,
        uniformtext_mode="show",
        updatemenus=[
            play_pause_menu(
                frame_names_by_horizon[
                    default_horizon
                ]
            )
        ],
        sliders=[
            {
                "active": default_active,
                "yanchor": "top",
                "xanchor": "left",
                "currentvalue": {
                    "visible": True,
                    "prefix": "Simulation horizon: ",
                    "font": {
                        "color": OFF_WHITE,
                        "size": 13,
                    },
                    "xanchor": "center",
                },
                "transition": {
                    "duration": 0
                },
                "pad": {
                    "b": 0,
                    "t": 40,
                },
                "len": 0.58,
                "x": 0.40,
                "y": -0.12,
                "bgcolor": TRANSPARENT,
                "bordercolor": "rgba(255,255,255,0.22)",
                "borderwidth": 1,
                "tickcolor": OFF_WHITE,
                "font": {
                    "color": OFF_WHITE
                },
                "steps": horizon_slider_steps,
            }
        ],
    )

    fig.update_annotations(
        font=dict(
            color=OFF_WHITE,
            size=15,
        )
    )

    # Upper axes.
    for col, scenario in enumerate(
        default_data["scenarios"],
        start=1,
    ):
        fig.update_xaxes(
            AXIS_STYLE,
            row=1,
            col=col,
            range=[
                default_data[
                    "dates"
                ][0],
                default_data[
                    "dates"
                ][-1],
            ],
            title_text="Date",
        )

        fig.update_yaxes(
            AXIS_STYLE,
            row=1,
            col=col,
            type="log",
            range=scenario["y_range"],
            title_text=(
                "Wealth multiple "
                "(log scale)"
            ),
            tickformat=".2~g",
        )

    # Lower retained/drag axes.
    for col in (1, 2):
        fig.update_xaxes(
            AXIS_STYLE,
            row=2,
            col=col,
            range=[
                0.0,
                return_x_max,
            ],
            ticksuffix="%",
            title_text="Annual return",
        )

        fig.update_yaxes(
            AXIS_STYLE,
            row=2,
            col=col,
            showgrid=False,
            title_text="",
        )

    return fig


# ============================================================
# Run — display only, with no HTML export
# ============================================================

if __name__ == "__main__":
    fig = build_figure(
        DEFAULT_HORIZON
    )
    fig.show()

*Here's the technicality: assuming the volatility doesn't lead to meaningful wealth creation, more on this in a moment*

###### ______________________________________________________________________________________________________________________________________

##### 🌊 The Only Thing Worse Than Volatility: *Needing Liquidity*

Downside variation can put you in highly undesirable states of the world

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# Config
# ============================================================

SEED = 11
rng = np.random.default_rng(SEED)

YEARS = 15
STEPS_PER_YEAR = 12
N_STEPS = YEARS * STEPS_PER_YEAR
DT = 1.0 / STEPS_PER_YEAR

INITIAL_VALUE = 1.0
START_DATE = "2025-01-01"

FRAME_STRIDE = 3
FRAME_DURATION = 60
INITIAL_I = 3

CRASH_YEAR = 8.0
CRASH_STEP = int(CRASH_YEAR * STEPS_PER_YEAR)

PUT_MONETIZATION_MONTHS = 3
POST_CRASH_RECOVERY_MONTHS = 6

SHOW_FIG = True

N_ALT_PATHS = 14  # Number of alternative, plausible-looking paths (background)

# ============================================================
# Portfolio assumptions
# ============================================================

PORTFOLIOS = [
    {
        "key": "regular",
        "title": "Regular Portfolio",
        "mu": 0.080,
        "sigma": 0.150,
        "gap_down": -0.25,
        "monetization_bonus": 0.00,
        "recovery_bonus": 0.03,
        "path_color": "#4da3ff",
        "subtitle": "25% gap-down, then resumes uptrend",
    },
    {
        "key": "hedge",
        "title": "Hedge Portfolio",
        "mu": 0.095,
        "sigma": 0.100,
        "gap_down": -0.10,
        "monetization_bonus": 0.07,
        "recovery_bonus": 0.03,
        "path_color": "#7ee081",
        "subtitle": "10% gap-down, monetizes puts, then resumes uptrend",
    },
]

# ============================================================
# Styling
# ============================================================

TRANSPARENT = "rgba(0,0,0,0)"

OFF_WHITE = "#e0e0e0"
MUTED_WHITE = "#b8b8b8"

ALT_PATH_COLOR = "rgba(120,120,120,0.23)"
ALT_PATH_WIDTH = 1.05  # Slightly wider for more "sample path" bolding

MAIN_PATH_WIDTH = 3.2

GEOMETRIC_RETURN_COLOR = "#22e97b"
VOLATILITY_DRAG_COLOR = "#fd6262"

BASELINE_COLOR = "#777777"
CRASH_LINE_COLOR = "#ff8a8a"
CRASH_BAND_FILL = "rgba(255,80,80,0.10)"
MONETIZATION_FILL = "rgba(126,224,129,0.12)"

AXIS_STYLE = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.10)",
    tickfont=dict(color=OFF_WHITE),
    linecolor=OFF_WHITE,
    zeroline=False,
    title_font=dict(color=OFF_WHITE),
)

BAR_LABEL_FONT = dict(color="#111111", size=13, family="Roboto,sans-serif")

# ============================================================
# Helpers
# ============================================================

def padded_range(values, pad_fraction=0.08, min_pad=0.12, floor=None):
    values = np.asarray(values, dtype=float)
    v_min = float(np.nanmin(values))
    v_max = float(np.nanmax(values))

    if np.isclose(v_min, v_max):
        pad = max(abs(v_max) * pad_fraction, min_pad)
    else:
        pad = max((v_max - v_min) * pad_fraction, min_pad)

    lower = v_min - pad
    upper = v_max + pad
    if floor is not None:
        lower = max(floor, lower)
    return [lower, upper]

def build_single_path(
    s0,
    mu,
    sigma,
    gap_down,
    monetization_bonus,
    recovery_bonus,
    crash_step,
    rng,
):
    month_idx = np.arange(N_STEPS)
    base = (mu - 0.5 * sigma**2) * DT
    cycle = 0.18 * sigma * np.sin(2.0 * np.pi * month_idx / 30.0) * np.sqrt(DT)
    noise = 0.16 * sigma * np.sqrt(DT) * rng.normal(size=N_STEPS)
    log_returns = base + cycle + noise

    # Crash gap.
    log_returns[crash_step] += np.log(1.0 + gap_down)
    # Market resumes uptrend after the drawdown.
    if recovery_bonus > 0.0:
        recovery_slice = slice(
            crash_step + 1,
            min(crash_step + 1 + POST_CRASH_RECOVERY_MONTHS, N_STEPS),
        )
        n = recovery_slice.stop - recovery_slice.start
        if n > 0:
            log_returns[recovery_slice] += np.log(1.0 + recovery_bonus) / n
    # Hedge monetizes puts during / after the drawdown.
    if monetization_bonus > 0.0:
        monetization_slice = slice(
            crash_step + 1,
            min(crash_step + 1 + PUT_MONETIZATION_MONTHS, N_STEPS),
        )
        n = monetization_slice.stop - monetization_slice.start
        if n > 0:
            log_returns[monetization_slice] += np.log(1.0 + monetization_bonus) / n
    return s0 * np.exp(np.r_[0.0, np.cumsum(log_returns)])

def build_alt_paths(s0, mu, sigma, gap_down, monetization_bonus, recovery_bonus, crash_step, n_paths, rng):
    # More randomness for more realistic sample paths
    alt_paths = []
    for i in range(n_paths):
        # Jitter more strongly for increased randomness
        mu_jitter = mu + rng.normal(0, 0.010)           # Was 0.004
        sigma_jitter = sigma * rng.uniform(0.9, 1.20)    # Was 0.002 noise, now up to 20% swing
        monetization_jitter = monetization_bonus * rng.uniform(0.7, 1.23)   # Larger
        recovery_jitter = recovery_bonus * rng.uniform(0.75, 1.2)
        crash_step_jitter = int(crash_step + rng.integers(-6, 7))           # Much more variable
        gap_down_jitter = gap_down + rng.normal(0, 0.035)                   # Larger random crash size
        # Noise in initial value to "fan out" curves
        init_value_jitter = s0 * rng.uniform(0.98, 1.04)
        alt_rng = np.random.default_rng(rng.integers(0, 1e9))
        path = build_single_path(
            init_value_jitter,
            mu_jitter,
            sigma_jitter,
            gap_down_jitter,
            monetization_jitter,
            recovery_jitter,
            crash_step_jitter,
            alt_rng
        )
        # Add final "noise blur" to exaggerate divergence
        if rng.random() < 0.7:
            path *= np.exp(rng.normal(0, 0.009 * (np.arange(len(path)) / len(path)), size=len(path)))
        alt_paths.append(path)
    return np.stack(alt_paths, axis=1) # shape: (n_points, n_paths)

def make_portfolio_main_trace(dates, path_values, portfolio_name, path_color):
    n_points = len(path_values)
    return go.Scatter(
        x=dates,
        y=path_values,
        mode="lines",
        line=dict(color=path_color, width=MAIN_PATH_WIDTH),
        showlegend=False,
        hovertemplate=(
            f"{portfolio_name}<br>"
            "Date: %{x|%Y-%m-%d}<br>"
            "Wealth: %{y:.4f}×<extra></extra>"
        ),
    )

def make_portfolio_alt_trace(dates, alt_values, path_color):
    # alt_values shape: (n_points, n_paths)
    traces = []
    n_paths = alt_values.shape[1]
    for j in range(n_paths):
        # Use a wide color gray with some alpha, just to help with the "fuzzy" sample path visual
        traces.append(
            go.Scatter(
                x=dates,
                y=alt_values[:, j],
                mode="lines",
                line=dict(
                    color=ALT_PATH_COLOR,
                    width=ALT_PATH_WIDTH,
                    shape="spline" if np.random.rand() > 0.75 else "linear"  # Add a few soft curves
                ),
                hoverinfo="skip",
                showlegend=False,
            )
        )
    return traces

def make_bottom_bar_traces(portfolio, label_prefix=None):
    """
    Single column:
        - retained geometric return
        - volatility drag
    Styled split-bar for clarity.
    """
    arithmetic_pct = 100.0 * portfolio["mu"]
    drag_pct = 100.0 * 0.5 * portfolio["sigma"]**2
    retained_pct = arithmetic_pct - drag_pct

    # decide proper label prefix
    if label_prefix is None:
        prefix_str = f"{portfolio['title']}<br>"
    else:
        prefix_str = f"{label_prefix}<br>"

    # Retained bar
    retained = go.Bar(
        x=[retained_pct],
        y=["Arithmetic return"],
        base=[0.0],
        orientation="h",
        width=0.7,
        marker=dict(
            color=GEOMETRIC_RETURN_COLOR,
            line=dict(width=2, color="rgba(0,0,0,0.03)"),
        ),
        showlegend=False,
        hovertemplate=(
            prefix_str +
            f"<span style='color:#111111'><b>Retained CAGR:</b> {retained_pct:.2f}%</span><extra></extra>"
        ),
        customdata=[["CAGR", f"{retained_pct:.2f}%"]],
        text=[f"{retained_pct:.2f}%"],
        textfont=BAR_LABEL_FONT,
        textposition="inside",
        insidetextanchor="middle",
    )
    # Drag bar
    drag = go.Bar(
        x=[drag_pct],
        y=["Arithmetic return"],
        base=[retained_pct],
        orientation="h",
        width=0.7,
        marker=dict(
            color=VOLATILITY_DRAG_COLOR,
            line=dict(width=2, color="rgba(0,0,0,0.03)"),
        ),
        showlegend=False,
        hovertemplate=(
            prefix_str +
            f"<span style='color:#111111'><b>Volatility drag:</b> {drag_pct:.2f}%</span><extra></extra>"
        ),
        customdata=[["Drag", f"{drag_pct:.2f}%"]],
        text=[f"{drag_pct:.2f}%"],
        textfont=BAR_LABEL_FONT,
        textposition="inside",
        insidetextanchor="middle",
    )
    return retained, drag

def figure_title(step, portfolios, dates):
    elapsed_years = step / STEPS_PER_YEAR
    crash_date = dates[CRASH_STEP].strftime("%Y-%m")
    return (
        "Hedged drawdown management: lower downside variation reduces volatility drag"
        "<br><sup>"
        f"Elapsed: {elapsed_years:.1f} years"
        f" &nbsp;|&nbsp; Crash event at {crash_date}"
        " &nbsp;|&nbsp; Regular: -25% gap"
        " &nbsp;|&nbsp; Hedge: -10% gap + put monetization"
        "</sup>"
    )

# ============================================================
# Time axis, portfolio + alternative paths
# ============================================================

dates = pd.date_range(
    start=START_DATE,
    periods=N_STEPS + 1,
    freq="MS",
)

# Prep main paths & alternate backgrounds
portfolio_rngs = {
    "regular": np.random.default_rng(SEED + 101),
    "hedge": np.random.default_rng(SEED + 202),
}

for portfolio in PORTFOLIOS:
    portfolio["drag"] = 0.5 * portfolio["sigma"]**2
    portfolio["geometric"] = portfolio["mu"] - portfolio["drag"]
    portfolio["arithmetic_pct"] = 100.0 * portfolio["mu"]
    portfolio["drag_pct"] = 100.0 * portfolio["drag"]
    portfolio["geometric_pct"] = 100.0 * portfolio["geometric"]
    portfolio["main_path"] = build_single_path(
        s0=INITIAL_VALUE,
        mu=portfolio["mu"],
        sigma=portfolio["sigma"],
        gap_down=portfolio["gap_down"],
        monetization_bonus=portfolio["monetization_bonus"],
        recovery_bonus=portfolio["recovery_bonus"],
        crash_step=CRASH_STEP,
        rng=portfolio_rngs[portfolio["key"]],
    )
    # Generate alt paths, but always with SEED offset for consistency
    portfolio["alt_paths"] = build_alt_paths(
        s0=INITIAL_VALUE,
        mu=portfolio["mu"],
        sigma=portfolio["sigma"],
        gap_down=portfolio["gap_down"],
        monetization_bonus=portfolio["monetization_bonus"],
        recovery_bonus=portfolio["recovery_bonus"],
        crash_step=CRASH_STEP,
        n_paths=N_ALT_PATHS,
        rng=np.random.default_rng(SEED + 3000 + (201 if portfolio["key"] == "hedge" else 101))
    )

# Common y axis
top_y_range = padded_range(
    np.r_[
        PORTFOLIOS[0]["main_path"],
        PORTFOLIOS[1]["main_path"],
        PORTFOLIOS[0]["alt_paths"].flatten(),
        PORTFOLIOS[1]["alt_paths"].flatten(),
    ],
    pad_fraction=0.08,
    min_pad=0.18,
    floor=0.45,
)

return_x_max = 1.18 * max(p["arithmetic_pct"] for p in PORTFOLIOS)

frame_indices = list(range(INITIAL_I, N_STEPS + 1, FRAME_STRIDE))
if frame_indices[-1] != N_STEPS:
    frame_indices.append(N_STEPS)
initial_end = min(INITIAL_I, N_STEPS)
initial_n_points = initial_end + 1

# ============================================================
# Figure layout
# ============================================================

subplot_titles = (
    f"{PORTFOLIOS[0]['title']}"
    "<br><sup>"
    f"{PORTFOLIOS[0]['subtitle']}"
    f" &nbsp;|&nbsp; μ = {PORTFOLIOS[0]['mu']:.1%}, "
    f"σ = {PORTFOLIOS[0]['sigma']:.1%}, "
    f"CAGR ≈ {PORTFOLIOS[0]['geometric']:.1%}"
    "</sup>",

    f"{PORTFOLIOS[1]['title']}"
    "<br><sup>"
    f"{PORTFOLIOS[1]['subtitle']}"
    f" &nbsp;|&nbsp; μ = {PORTFOLIOS[1]['mu']:.1%}, "
    f"σ = {PORTFOLIOS[1]['sigma']:.1%}, "
    f"CAGR ≈ {PORTFOLIOS[1]['geometric']:.1%}"
    "</sup>",

    "<span style='font-size:1.14em;font-variant:small-caps'>Volatility drag decomposition</span>",
    "<span style='font-size:1.14em;font-variant:small-caps'>Volatility drag decomposition</span>",
)

fig = make_subplots(
    rows=2,
    cols=2,
    row_heights=[0.72, 0.28],
    column_widths=[0.50, 0.50],
    horizontal_spacing=0.08,
    vertical_spacing=0.18,
    subplot_titles=subplot_titles,
)

# ============================================================
# Initial top-row traces
# ============================================================

for col, portfolio in enumerate(PORTFOLIOS, start=1):
    dates_now = dates[:initial_n_points]
    # Plot all alt paths
    alt_traces = make_portfolio_alt_trace(dates_now, portfolio["alt_paths"][:initial_n_points, :], ALT_PATH_COLOR)
    for t in alt_traces:
        fig.add_trace(t, row=1, col=col)
    # Plot main path
    fig.add_trace(
        make_portfolio_main_trace(
            dates_now,
            portfolio["main_path"][:initial_n_points],
            portfolio["title"],
            portfolio["path_color"],
        ),
        row=1,
        col=col,
    )

# ============================================================
# Initial lower-panel traces
# ============================================================

# Only plot the green/pink bar stacks, no 'arithmetic total' line or annotation
reg_retained, reg_drag = make_bottom_bar_traces(PORTFOLIOS[0])
hed_retained, hed_drag = make_bottom_bar_traces(PORTFOLIOS[1])

fig.add_trace(reg_retained, row=2, col=1)
fig.add_trace(reg_drag, row=2, col=1)
fig.add_trace(hed_retained, row=2, col=2)
fig.add_trace(hed_drag, row=2, col=2)

# (Removed: black tick lines & annotations for arithmetic total)

# ============================================================
# Baselines and event markers (no overlay text annotations)
# ============================================================

for col, portfolio in enumerate(PORTFOLIOS, start=1):
    fig.add_hline(
        y=INITIAL_VALUE,
        line=dict(color=BASELINE_COLOR, width=1, dash="dash"),
        opacity=0.65,
        row=1,
        col=col,
    )
    # Crash band
    fig.add_vrect(
        x0=dates[CRASH_STEP],
        x1=dates[min(CRASH_STEP + 1, N_STEPS)],
        fillcolor=CRASH_BAND_FILL,
        line_width=0,
        row=1,
        col=col,
    )
    # Crash line
    fig.add_vline(
        x=dates[CRASH_STEP],
        line=dict(color=CRASH_LINE_COLOR, width=1.8, dash="dot"),
        opacity=0.85,
        row=1,
        col=col,
    )
# Monetization window highlight
fig.add_vrect(
    x0=dates[CRASH_STEP + 1],
    x1=dates[min(CRASH_STEP + PUT_MONETIZATION_MONTHS, N_STEPS)],
    fillcolor=MONETIZATION_FILL,
    line_width=0,
    row=1,
    col=2,
)

# ============================================================
# Animation frames
# ============================================================

frames = []
slider_steps = []
n_alt_traces_per_col = N_ALT_PATHS

n_traces_toprow = (N_ALT_PATHS + 1) * 2

frame_trace_order = []
for col in range(2):
    frame_trace_order.extend([None] * N_ALT_PATHS)   # alt paths col X
    frame_trace_order.append(None)                   # main path

frame_trace_order.extend([None, None, None, None])    # 4 bar chart traces

for i in frame_indices:
    n_points = i + 1
    frame_name = f"f{i}"
    frame_data = []
    for portfolio in PORTFOLIOS:
        dates_now = dates[:n_points]
        # Add alt paths
        for j in range(N_ALT_PATHS):
            # new extra squiggliness for animation too!
            alt_path = portfolio["alt_paths"][:n_points, j]
            # Add a little more noise for each path slice to remain visually distinct
            if n_points > 10 and np.random.rand() > 0.35:
                alt_path = alt_path * np.exp(np.random.normal(0, 0.008 * (np.arange(n_points) / n_points), size=n_points))
            frame_data.append(
                go.Scatter(
                    x=dates_now,
                    y=alt_path,
                    mode="lines",
                    line=dict(color=ALT_PATH_COLOR, width=ALT_PATH_WIDTH, shape="spline" if np.random.rand() > 0.75 else "linear"),
                    hoverinfo="skip",
                    showlegend=False,
                )
            )
        # Add main
        frame_data.append(
            make_portfolio_main_trace(
                dates_now,
                portfolio["main_path"][:n_points],
                portfolio["title"],
                portfolio["path_color"],
            )
        )
    # Keep bars fixed in animation
    frame_data.extend([
        reg_retained,
        reg_drag,
        hed_retained,
        hed_drag,
    ])

    frames.append(
        go.Frame(
            data=frame_data,
            traces=list(range(n_traces_toprow + 4)),
            name=frame_name,
            layout=go.Layout(
                title=dict(
                    text=figure_title(i, PORTFOLIOS, dates),
                    font=dict(color=OFF_WHITE),                 # Chart title & subtitle: white
                )
            ),
        )
    )
    elapsed_years = i / STEPS_PER_YEAR
    slider_steps.append(
        {
            "args": [
                [frame_name],
                {
                    "frame": {"duration": 0, "redraw": True},
                    "mode": "immediate",
                    "fromcurrent": True,
                    "transition": {"duration": 0},
                },
            ],
            "label": f"{elapsed_years:.1f}Y",
            "method": "animate",
        }
    )

fig.frames = frames

# ============================================================
# Layout
# ============================================================

fig.update_layout(
    title=dict(
        text=figure_title(initial_end, PORTFOLIOS, dates),
        x=0.5,
        font=dict(color=OFF_WHITE),  # Chart title & subtitle: white
    ),
    template="plotly_dark",
    paper_bgcolor=TRANSPARENT,
    plot_bgcolor=TRANSPARENT,
    font=dict(color=OFF_WHITE),  # UI and subtitle: white
    height=800,
    width=1200,
    margin=dict(t=125, b=130, r=55, l=85),
    barmode="relative",
    bargap=0.15,
    hovermode="closest",
    showlegend=False,
    updatemenus=[
        {
            "type": "buttons",
            "buttons": [
                {
                    "label": "▶ Play",
                    "method": "animate",
                    "args": [
                        None,
                        {
                            "frame": {
                                "duration": FRAME_DURATION,
                                "redraw": True,
                            },
                            "transition": {"duration": 0},
                            "fromcurrent": True,
                        },
                    ],
                },
                {
                    "label": "⏸ Pause",
                    "method": "animate",
                    "args": [
                        [None],
                        {
                            "frame": {"duration": 0, "redraw": True},
                            "mode": "immediate",
                            "fromcurrent": True,
                        },
                    ],
                },
            ],
            "direction": "left",
            "pad": {"r": 10, "t": 70},
            "showactive": False,
            "x": 0.10,
            "xanchor": "right",
            "y": -0.03,
            "yanchor": "top",
            "bgcolor": TRANSPARENT,
            "bordercolor": "rgba(255,255,255,0.22)",
            "borderwidth": 1,
            "font": {"color": OFF_WHITE},
        }
    ],
    sliders=[
        {
            "active": 0,
            "yanchor": "top",
            "xanchor": "left",
            "currentvalue": {"visible": False},
            "transition": {"duration": 0},
            "pad": {"b": 10, "t": 40},
            "len": 0.85,
            "x": 0.15,
            "y": -0.03,
            "bgcolor": TRANSPARENT,
            "bordercolor": "rgba(255,255,255,0.22)",
            "borderwidth": 1,
            "tickcolor": OFF_WHITE,
            "font": {"color": OFF_WHITE},
            "steps": slider_steps,
        }
    ],
)

# Set all subplot annotation font color to white EXCEPT for lower bar chart overlays, which remain black (set via BAR_LABEL_FONT)
for i, annot in enumerate(fig.layout.annotations):
    # Only change color for top-row (chart title/subtitles) and barchart titles/subtitles (not overlays)
    if i < 4:  # 4 subplot annotation titles
        fig.layout.annotations[i].font.color = OFF_WHITE
    # (Tooltips/text on bars remain explicit black, specified in BAR_LABEL_FONT, so no need to change.)

# ============================================================
# Axes
# ============================================================

# Top row axes
for col in (1, 2):
    fig.update_xaxes(
        AXIS_STYLE,
        row=1,
        col=col,
        range=[dates[0], dates[-1]],
        title_text="Date",
    )
    fig.update_yaxes(
        AXIS_STYLE,
        row=1,
        col=col,
        range=top_y_range,
        title_text="Wealth multiple",
    )

# Bottom row axes
for col in (1, 2):
    fig.update_xaxes(
        AXIS_STYLE,
        row=2,
        col=col,
        range=[0.0, return_x_max],
        ticksuffix="%",
        title_text="Annual return",
    )
    fig.update_yaxes(
        AXIS_STYLE,
        row=2,
        col=col,
        showgrid=False,
        title_text="",
    )

# ============================================================
# Display only — no HTML export
# ============================================================

if SHOW_FIG:
    fig.show()

###### ______________________________________________________________________________________________________________________________________

##### 📈 The Confusing Part About Volatility Drag

Interestingly, (and confusingly) there is *good* and *bad* volatility drag

This can happen in one of a variety of different states:

    1.) Big trade with a positive payoff

    2.) Buying the fire sale

    3.) Holding a volatility "diversified" portfolio 

But remember, there's no free lunch and you don't get something for nothing

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# Config
# ============================================================

INITIAL_VALUE = 1.0
START_DATE = "2025-01-01"
YEARS = 1
STEPS_PER_YEAR = 12
N_STEPS = YEARS * STEPS_PER_YEAR
DT = 1 / STEPS_PER_YEAR
N_PATHS = 18

FRAME_STRIDE = 1
FRAME_DURATION = 260
INITIAL_I = 0

STRATEGIES = {
    "slow": {
        "label": "Slow growth",
        "mu": 0.10,
        "sigma": 0.20,
        "realized_return": 0.10,
        "path_seed": 101,
        "actual_seed": 1,
        "actual_color": "#18d618",
    },
    "chase": {
        "label": "Ephemeral return chase",
        "mu": 0.40,
        "sigma": 0.60,
        "realized_return": 0.40,
        "path_seed": 218,
        "actual_seed": 5,
        "actual_color": "#ffd84d",
    },
}

# ============================================================
# Simulation and calculation helpers
# ============================================================


def simulate_gbm_paths(s0, mu, sigma, n_steps, n_paths, seed):
    rng = np.random.default_rng(seed)
    z = rng.normal(size=(n_steps, n_paths))
    log_returns = (
        (mu - 0.5 * sigma**2) * DT
        + sigma * np.sqrt(DT) * z
    )
    cumulative_log_returns = np.vstack([
        np.zeros(n_paths),
        np.cumsum(log_returns, axis=0),
    ])
    return s0 * np.exp(cumulative_log_returns)


def simulate_conditioned_trade_path(s0, realized_return, sigma, n_steps, seed):
    rng = np.random.default_rng(seed)
    z = rng.normal(size=n_steps)
    local_times = np.arange(n_steps + 1) * DT
    horizon = local_times[-1]
    brownian_motion = np.r_[
        0.0,
        np.cumsum(np.sqrt(DT) * z),
    ]
    brownian_bridge = (
        brownian_motion
        - (local_times / horizon) * brownian_motion[-1]
    )
    target_log_return = np.log1p(realized_return)
    conditioned_log_path = (
        (local_times / horizon) * target_log_return
        + sigma * brownian_bridge
    )
    return s0 * np.exp(conditioned_log_path)


def expected_geometric_path(s0, mu, sigma, times):
    return s0 * np.exp((mu - 0.5 * sigma**2) * times)


def return_decomposition(mu, sigma):
    arithmetic_pct = 100.0 * mu
    drag_pct = 100.0 * 0.5 * sigma**2
    retained_pct = arithmetic_pct - drag_pct
    return arithmetic_pct, drag_pct, retained_pct


def padded_range(values, pad_fraction=0.08, min_pad=0.08):
    values = np.asarray(values, dtype=float)
    v_min = float(np.nanmin(values))
    v_max = float(np.nanmax(values))
    if np.isclose(v_min, v_max):
        pad = max(abs(v_max) * pad_fraction, min_pad)
    else:
        pad = max((v_max - v_min) * pad_fraction, min_pad)
    return [max(0.0, v_min - pad), v_max + pad]


def cumulative_return_pct(value, initial_value=INITIAL_VALUE):
    return 100.0 * (value / initial_value - 1.0)


# ============================================================
# Data
# ============================================================

dates = pd.date_range(
    start=START_DATE,
    periods=N_STEPS + 1,
    freq="MS",
)
times = np.arange(N_STEPS + 1) * DT

for strategy in STRATEGIES.values():
    strategy["paths"] = simulate_gbm_paths(
        s0=INITIAL_VALUE,
        mu=strategy["mu"],
        sigma=strategy["sigma"],
        n_steps=N_STEPS,
        n_paths=N_PATHS,
        seed=strategy["path_seed"],
    )
    strategy["actual_path"] = simulate_conditioned_trade_path(
        s0=INITIAL_VALUE,
        realized_return=strategy["realized_return"],
        sigma=strategy["sigma"],
        n_steps=N_STEPS,
        seed=strategy["actual_seed"],
    )
    strategy["geometric_path"] = expected_geometric_path(
        INITIAL_VALUE,
        strategy["mu"],
        strategy["sigma"],
        times,
    )
    (
        strategy["arithmetic_pct"],
        strategy["drag_pct"],
        strategy["retained_pct"],
    ) = return_decomposition(strategy["mu"], strategy["sigma"])

slow = STRATEGIES["slow"]
chase = STRATEGIES["chase"]

all_top_values = np.r_[
    slow["paths"].ravel(),
    chase["paths"].ravel(),
    slow["actual_path"],
    chase["actual_path"],
    slow["geometric_path"],
    chase["geometric_path"],
]
top_y_range = padded_range(all_top_values, pad_fraction=0.06, min_pad=0.10)
return_x_max = max(
    slow["arithmetic_pct"],
    chase["arithmetic_pct"],
) * 1.16

# ============================================================
# Styling
# ============================================================

off_white = "#e0e0e0"
cloud_color = "rgba(224,224,224,0.26)"
expected_path_color = "rgba(0,255,136,0.72)"
volatility_drag_color = "#ff3030"
retained_return_color = "#18d618"
baseline_color = "#777777"

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.10)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white),
)

# ============================================================
# Bottom-bar trace factories
# ============================================================

BAR_LABELS = ["Slow growth", "Return chase"]
RETAINED_VALUES = [slow["retained_pct"], chase["retained_pct"]]
DRAG_VALUES = [slow["drag_pct"], chase["drag_pct"]]


def make_retained_bar():
    return go.Bar(
        x=RETAINED_VALUES,
        y=BAR_LABELS,
        base=[0.0, 0.0],
        orientation="h",
        width=0.52,
        marker=dict(color=retained_return_color),
        text=[f"Retained {value:.1f}%" for value in RETAINED_VALUES],
        textposition="inside",
        insidetextanchor="middle",
        textfont=dict(size=14),
        name="Expected geometric growth retained",
        showlegend=False,
        hovertemplate=(
            "%{y}<br>"
            "Expected geometric growth retained: %{x:.1f}%<extra></extra>"
        ),
    )


def make_drag_bar():
    return go.Bar(
        x=DRAG_VALUES,
        y=BAR_LABELS,
        base=RETAINED_VALUES,
        orientation="h",
        width=0.52,
        marker=dict(color=volatility_drag_color),
        text=[f"Drag {value:.1f}%" for value in DRAG_VALUES],
        textposition="inside",
        insidetextanchor="middle",
        textfont=dict(size=14),
        name="Volatility drag",
        showlegend=False,
        hovertemplate=(
            "%{y}<br>"
            "Volatility drag = 1/2 sigma squared: %{x:.1f}%<extra></extra>"
        ),
    )


# ============================================================
# Helper to format subplot titles with actual return
# (Customize: Style the return more like the attached plot)
# ============================================================

def styled_return_str(actual_return, color):
    prefix = "+" if actual_return >= 0 else ""
    return (
        f"<span style='color:{color};"
        "font-weight:bold; font-size:17px;"
        "font-family:Menlo, Inconsolata, monospace;'>"
        f"{prefix}{actual_return:.0f}%"
        "</span>"
    )

def format_subplot_title(label, path, color):
    actual_return = cumulative_return_pct(path[-1])
    return (
        f"{label}: Actual Trade Finishes "
        f"{styled_return_str(actual_return, color)}"
    )

# ============================================================
# Figure construction
# ============================================================

def build_figure():
    # Construct subplot titles with actual returns included, styled
    subplot_titles = [
        format_subplot_title(slow["label"], slow["actual_path"], slow["actual_color"]),
        format_subplot_title(chase["label"], chase["actual_path"], chase["actual_color"]),
        "Arithmetic Return and Volatility Drag",
    ]

    fig = make_subplots(
        rows=2,
        cols=2,
        specs=[
            [{"type": "xy"}, {"type": "xy"}],
            [{"type": "xy", "colspan": 2}, None],
        ],
        row_heights=[0.72, 0.28],
        column_widths=[0.50, 0.50],
        horizontal_spacing=0.08,
        vertical_spacing=0.20,
        subplot_titles=tuple(subplot_titles),
    )

    # Make sure all annotation titles are rendered with HTML
    for ann in fig['layout']['annotations']:
        ann['font'] = dict(color=off_white, size=17)
        ann['align'] = "left"
        ann['showarrow'] = False
        ann['xref'] = "paper"
        ann['yref'] = "paper"

    initial_end = min(INITIAL_I, N_STEPS)
    initial_n_points = initial_end + 1

    animated_trace_indices = []

    # --------------------------------------------------------
    # Top panels
    # --------------------------------------------------------

    for col, strategy in ((1, slow), (2, chase)):
        for path_index in range(N_PATHS):
            trace_index = len(fig.data)
            fig.add_trace(
                go.Scatter(
                    x=dates[:initial_n_points],
                    y=strategy["paths"][:initial_n_points, path_index],
                    mode="lines",
                    line=dict(
                        color=cloud_color,
                        width=1.25,
                        simplify=False,
                    ),
                    opacity=1.0,
                    name=f"{strategy['label']} simulation {path_index + 1}",
                    showlegend=False,
                    hovertemplate=(
                        f"{strategy['label']} simulation {path_index + 1}<br>"
                        "Date: %{x|%Y-%m-%d}<br>"
                        "Wealth: %{y:.3f}<extra></extra>"
                    ),
                ),
                row=1,
                col=col,
            )
            animated_trace_indices.append(trace_index)

        trace_index = len(fig.data)
        fig.add_trace(
            go.Scatter(
                x=dates[:initial_n_points],
                y=strategy["actual_path"][:initial_n_points],
                mode="lines",
                line=dict(
                    color=strategy["actual_color"],
                    width=5,
                    simplify=False,
                ),
                name=f"{strategy['label']} actual trade",
                showlegend=False,
                customdata=[
                    cumulative_return_pct(value)
                    for value in strategy["actual_path"][:initial_n_points]
                ],
                hovertemplate=(
                    f"{strategy['label']} actual trade<br>"
                    "Date: %{x|%Y-%m-%d}<br>"
                    "Wealth: %{y:.3f}<br>"
                    "Cumulative return: %{customdata:+.1f}%<extra></extra>"
                ),
            ),
            row=1,
            col=col,
        )
        animated_trace_indices.append(trace_index)

        # Remove "Actual XX%" text marker overlay for clarity (per prompt instruction)
        trace_index = len(fig.data)
        fig.add_trace(
            go.Scatter(
                x=[dates[initial_end]],
                y=[strategy["actual_path"][initial_end]],
                mode="markers",
                marker=dict(
                    color=strategy["actual_color"],
                    size=11,
                    line=dict(color=off_white, width=1),
                ),
                showlegend=False,
                hoverinfo="skip",
            ),
            row=1,
            col=col,
        )
        animated_trace_indices.append(trace_index)

        trace_index = len(fig.data)
        fig.add_trace(
            go.Scatter(
                x=dates[:initial_n_points],
                y=strategy["geometric_path"][:initial_n_points],
                mode="lines",
                line=dict(
                    color=expected_path_color,
                    width=2.5,
                    dash="dash",
                    simplify=False,
                ),
                name=f"{strategy['label']} expected compounded path",
                showlegend=False,
                hovertemplate=(
                    "Expected compounded path<br>"
                    "Date: %{x|%Y-%m-%d}<br>"
                    "Wealth: %{y:.3f}<extra></extra>"
                ),
            ),
            row=1,
            col=col,
        )
        animated_trace_indices.append(trace_index)

        fig.add_hline(
            y=INITIAL_VALUE,
            line=dict(color=baseline_color, width=1, dash="dot"),
            opacity=0.80,
            row=1,
            col=col,
        )

    # --------------------------------------------------------
    # Bottom panel
    # --------------------------------------------------------

    bottom_retained_index = len(fig.data)
    fig.add_trace(make_retained_bar(), row=2, col=1)

    bottom_drag_index = len(fig.data)
    fig.add_trace(make_drag_bar(), row=2, col=1)

    # --------------------------------------------------------
    # Animation frames
    # --------------------------------------------------------

    frames = []
    slider_steps = []

    frame_indices = list(range(initial_end, N_STEPS + 1, FRAME_STRIDE))
    if frame_indices[-1] != N_STEPS:
        frame_indices.append(N_STEPS)

    frame_trace_indices = animated_trace_indices + [
        bottom_retained_index,
        bottom_drag_index,
    ]

    for i in frame_indices:
        frame_name = f"f{i}"
        n_points = i + 1
        frame_data = []

        for strategy in (slow, chase):
            for path_index in range(N_PATHS):
                frame_data.append(
                    go.Scatter(
                        x=dates[:n_points],
                        y=strategy["paths"][:n_points, path_index],
                        mode="lines",
                        line=dict(
                            color=cloud_color,
                            width=1.25,
                            simplify=False,
                        ),
                        opacity=1.0,
                        showlegend=False,
                    )
                )

            frame_data.append(
                go.Scatter(
                    x=dates[:n_points],
                    y=strategy["actual_path"][:n_points],
                    mode="lines",
                    line=dict(
                        color=strategy["actual_color"],
                        width=5,
                        simplify=False,
                    ),
                    customdata=[
                        cumulative_return_pct(value)
                        for value in strategy["actual_path"][:n_points]
                    ],
                    showlegend=False,
                )
            )

            frame_data.append(
                go.Scatter(
                    x=[dates[i]],
                    y=[strategy["actual_path"][i]],
                    mode="markers",
                    marker=dict(
                        color=strategy["actual_color"],
                        size=11,
                        line=dict(color=off_white, width=1),
                    ),
                    showlegend=False,
                    hoverinfo="skip",
                )
            )

            frame_data.append(
                go.Scatter(
                    x=dates[:n_points],
                    y=strategy["geometric_path"][:n_points],
                    mode="lines",
                    line=dict(
                        color=expected_path_color,
                        width=2.5,
                        dash="dash",
                        simplify=False,
                    ),
                    showlegend=False,
                )
            )

        frame_data.append(make_retained_bar())
        frame_data.append(make_drag_bar())

        frames.append(
            go.Frame(
                data=frame_data,
                traces=frame_trace_indices,
                name=frame_name,
            )
        )

        slider_steps.append({
            "args": [
                [frame_name],
                {
                    "frame": {"duration": 0, "redraw": True},
                    "mode": "immediate",
                    "transition": {"duration": 0},
                    "fromcurrent": True,
                },
            ],
            "label": str(i),
            "method": "animate",
        })

    fig.frames = frames

    # --------------------------------------------------------
    # Layout
    # --------------------------------------------------------

    fig.update_layout(
        title=dict(
            text="Why Chasing the Big Trade Looks Rational — At First",
            x=0.5,
            font=dict(color=off_white, size=24),
        ),
        template="plotly_dark",
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        height=800,
        width=1200,
        margin=dict(t=120, b=125, r=65, l=95),
        barmode="overlay",
        bargap=0.20,
        hovermode="closest",
        showlegend=False,
        updatemenus=[{
            "type": "buttons",
            "buttons": [
                {
                    "label": "▶ Play",
                    "method": "animate",
                    "args": [
                        None,
                        {
                            "frame": {
                                "duration": FRAME_DURATION,
                                "redraw": True,
                            },
                            "transition": {"duration": 0},
                            "fromcurrent": True,
                        },
                    ],
                },
                {
                    "label": "⏸ Pause",
                    "method": "animate",
                    "args": [
                        [None],
                        {
                            "frame": {"duration": 0, "redraw": True},
                            "mode": "immediate",
                            "transition": {"duration": 0},
                            "fromcurrent": True,
                        },
                    ],
                },
            ],
            "direction": "left",
            "pad": {"r": 10, "t": 72},
            "showactive": False,
            "x": 0.10,
            "xanchor": "right",
            "y": -0.03,
            "yanchor": "top",
        }],
        sliders=[{
            "active": 0,
            "yanchor": "top",
            "xanchor": "left",
            "currentvalue": {"visible": False},
            "transition": {"duration": 0},
            "pad": {"b": 10, "t": 42},
            "len": 0.84,
            "x": 0.15,
            "y": -0.03,
            "steps": slider_steps,
        }],
    )

    # ----- Style subplot titles that contain HTML -----
    # This requires Plotly >= 5.5.0 and allow HTML in subplot_titles
    for ann in fig['layout']['annotations']:
        ann['font'] = dict(color=off_white, size=17)
        ann['align'] = "left"
        ann['showarrow'] = False
        ann['xref'] = "paper"
        ann['yref'] = "paper"
        ann['font']['family'] = "Menlo, Inconsolata, monospace"
        ann['captureevents'] = True
        ann['textangle'] = 0
        ann['bgcolor'] = None

    for col in (1, 2):
        fig.update_xaxes(
            axis_style,
            row=1,
            col=col,
            range=[dates[0], dates[-1]],
            title_text="Date",
        )
        fig.update_yaxes(
            axis_style,
            row=1,
            col=col,
            range=top_y_range,
            title_text="Wealth multiple" if col == 1 else "",
        )

    fig.update_xaxes(
        axis_style,
        row=2,
        col=1,
        range=[0, return_x_max],
        ticksuffix="%",
        title_text="Annual arithmetic return",
    )
    fig.update_yaxes(
        axis_style,
        row=2,
        col=1,
        showgrid=False,
        title_text="",
        categoryorder="array",
        categoryarray=BAR_LABELS,
        autorange="reversed",
    )

    return fig


if __name__ == "__main__":
    build_figure().show()

###### ______________________________________________________________________________________________________________________________________

##### 🛣️ Long-Run Impact of Chasing Ephemeral Returns

For many investors, it isn't appropriate to concentrate risk if it isn't your profession and the consequence is volatility drag

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# Config
# ============================================================

INITIAL_VALUE = 1.0
START_DATE = "2025-01-01"
YEARS = 15
STEPS_PER_YEAR = 12
N_STEPS = YEARS * STEPS_PER_YEAR
DT = 1 / STEPS_PER_YEAR
N_PATHS = 18

FRAME_STRIDE = 3            # quarterly reveal
FRAME_DURATION = 90
INITIAL_I = 0

# Same realized shock path for both highlighted strategies.
# This makes the comparison about the strategy, not about cherry-picking.
ACTUAL_PATH_SEED = 1433

STRATEGIES = {
    "slow": {
        "label": "Conservative path",
        "mu": 0.10,
        "sigma": 0.20,
        "cloud_seed": 101,
        "actual_color": "#18d618",
    },
    "chase": {
        "label": "Return chase",
        "mu": 0.40,
        "sigma": 0.95,
        "cloud_seed": 202,
        "actual_color": "#ffd84d",
    },
}

# ============================================================
# Helpers
# ============================================================


def simulate_gbm_paths(s0, mu, sigma, n_steps, n_paths, seed):
    rng = np.random.default_rng(seed)
    z = rng.normal(size=(n_steps, n_paths))
    log_returns = (
        (mu - 0.5 * sigma**2) * DT
        + sigma * np.sqrt(DT) * z
    )
    cumulative_log_returns = np.vstack([
        np.zeros(n_paths),
        np.cumsum(log_returns, axis=0),
    ])
    return s0 * np.exp(cumulative_log_returns)


def simulate_gbm_path_from_seed(s0, mu, sigma, n_steps, seed):
    rng = np.random.default_rng(seed)
    z = rng.normal(size=n_steps)
    log_returns = (
        (mu - 0.5 * sigma**2) * DT
        + sigma * np.sqrt(DT) * z
    )
    return s0 * np.exp(np.r_[0.0, np.cumsum(log_returns)])


def expected_geometric_path(s0, mu, sigma, times):
    return s0 * np.exp((mu - 0.5 * sigma**2) * times)


def return_decomposition(mu, sigma):
    arithmetic_pct = 100.0 * mu
    drag_pct = 100.0 * 0.5 * sigma**2
    retained_pct = arithmetic_pct - drag_pct
    return arithmetic_pct, drag_pct, retained_pct


def cumulative_return_pct(value, initial_value=INITIAL_VALUE):
    return 100.0 * (value / initial_value - 1.0)


def padded_range(low, high, pad_fraction=0.06, min_pad=0.08):
    span = high - low
    pad = max(span * pad_fraction, min_pad)
    return [max(0.0, low - pad), high + pad]


def make_retained_bar(retained_values, labels, retained_return_color):
    return go.Bar(
        x=retained_values,
        y=labels,
        base=[0.0 for _ in labels],
        orientation="h",
        width=0.54,
        marker=dict(color=retained_return_color),
        text=[f"Retained {value:.1f}%" for value in retained_values],
        textposition="inside",
        insidetextanchor="middle",
        textfont=dict(size=15),
        name="Geometric return",
        showlegend=False,
        hovertemplate="%{y}<br>Geometric return: %{x:.1f}%<extra></extra>",
    )


def make_drag_bar(drag_values, base_values, labels, volatility_drag_color):
    return go.Bar(
        x=drag_values,
        y=labels,
        base=base_values,
        orientation="h",
        width=0.54,
        marker=dict(color=volatility_drag_color),
        text=[f"Drag {value:.1f}%" for value in drag_values],
        textposition="inside",
        insidetextanchor="middle",
        textfont=dict(size=15),
        name="Volatility drag",
        showlegend=False,
        hovertemplate="%{y}<br>Volatility drag: %{x:.1f}%<extra></extra>",
    )


# ============================================================
# Data
# ============================================================

dates = pd.date_range(
    start=START_DATE,
    periods=N_STEPS + 1,
    freq="MS",
)
times = np.arange(N_STEPS + 1) * DT

for strategy in STRATEGIES.values():
    strategy["paths"] = simulate_gbm_paths(
        s0=INITIAL_VALUE,
        mu=strategy["mu"],
        sigma=strategy["sigma"],
        n_steps=N_STEPS,
        n_paths=N_PATHS,
        seed=strategy["cloud_seed"],
    )
    strategy["actual_path"] = simulate_gbm_path_from_seed(
        s0=INITIAL_VALUE,
        mu=strategy["mu"],
        sigma=strategy["sigma"],
        n_steps=N_STEPS,
        seed=ACTUAL_PATH_SEED,
    )
    strategy["geometric_path"] = expected_geometric_path(
        INITIAL_VALUE,
        strategy["mu"],
        strategy["sigma"],
        times,
    )
    (
        strategy["arithmetic_pct"],
        strategy["drag_pct"],
        strategy["retained_pct"],
    ) = return_decomposition(strategy["mu"], strategy["sigma"])

slow = STRATEGIES["slow"]
chase = STRATEGIES["chase"]

# The top-panel y-axis will be fixed to [0, 6] for both panels.
top_y_range = [0.0, 6.0]

bar_labels = ["Conservative", "Return chase"]
retained_values = [slow["retained_pct"], chase["retained_pct"]]
drag_values = [slow["drag_pct"], chase["drag_pct"]]
return_x_max = max(
    slow["arithmetic_pct"],
    chase["arithmetic_pct"],
    slow["retained_pct"] + slow["drag_pct"],
    chase["retained_pct"] + chase["drag_pct"],
) * 1.16

# ============================================================
# Styling
# ============================================================

off_white = "#e0e0e0"
cloud_color = "rgba(224,224,224,0.24)"
expected_path_color = "rgba(0,255,136,0.72)"
volatility_drag_color = "#ff3030"
retained_return_color = "#18d618"
baseline_color = "#777777"

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.10)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white),
)

# ============================================================
# Figure
# ============================================================

def build_figure():
    # Precompute actual cumulative returns for initial step (use t=0 for both strategies)
    initial_end = min(INITIAL_I, N_STEPS)
    initial_n_points = initial_end + 1

    # Instead of just cumulative return at t=0 (which is always 0%), show *current* return for each subplot 
    # based on the last point drawn for each subplot in the INITIAL frame
    # So here, we want the most recent return as far as actual_path is displayed

    # For the initial subplot_titles (static at frame 0): 
    # - As INITIAL_I is 0, initial_end is also 0. So "slow_actual_return" and "chase_actual_return" are both always 0.0.
    # - But in the animation, the correct return is displayed as the path animates.
    #
    # SOLUTION: Calculate the "final realized return" at the END (i.e. end of the path) to display in the titles at the *start*.
    # - But for initial frame, keep shown value as 0% (matches visible line state).
    # - For slider steps, titles are updated dynamically based on each path's most recent shown value.

    # We'll show the *ending cumulative return* in the chart titles at the very end (static), but in the animation,
    # the correct return dynamically as the path animates.

    # Final returns (as would be shown after full animation)
    final_slow_actual_return = cumulative_return_pct(slow["actual_path"][-1])
    final_chase_actual_return = cumulative_return_pct(chase["actual_path"][-1])

    # Titles for subplots with final actual return %
    subplot_titles = (
        f"Conservative Compounding<br><span style='font-size:0.92em;color:{slow['actual_color']}'>{final_slow_actual_return:+.1f}%</span>",
        f"High Return Chase with Extreme Volatility<br><span style='font-size:0.92em;color:{chase['actual_color']}'>{final_chase_actual_return:+.1f}%</span>",
        "Arithmetic Return and Volatility Drag",
    )

    fig = make_subplots(
        rows=2,
        cols=2,
        specs=[
            [{"type": "xy"}, {"type": "xy"}],
            [{"type": "xy", "colspan": 2}, None],
        ],
        row_heights=[0.72, 0.28],
        column_widths=[0.50, 0.50],
        horizontal_spacing=0.08,
        vertical_spacing=0.20,
        subplot_titles=subplot_titles,
    )

    animated_trace_indices = []

    for col, (strategy, _) in zip(
        (1, 2),
        ((slow, None), (chase, None))
    ):
        # Risk cloud.
        for path_index in range(N_PATHS):
            trace_index = len(fig.data)
            fig.add_trace(
                go.Scatter(
                    x=dates[:initial_n_points],
                    y=strategy["paths"][:initial_n_points, path_index],
                    mode="lines",
                    line=dict(color=cloud_color, width=1.2, simplify=False),
                    name=f"{strategy['label']} simulation {path_index + 1}",
                    showlegend=False,
                    hovertemplate=(
                        f"{strategy['label']} simulation {path_index + 1}<br>"
                        "Date: %{x|%Y-%m-%d}<br>"
                        "Wealth: %{y:.3f}<extra></extra>"
                    ),
                ),
                row=1,
                col=col,
            )
            animated_trace_indices.append(trace_index)

        # Highlighted realized path.
        trace_index = len(fig.data)
        fig.add_trace(
            go.Scatter(
                x=dates[:initial_n_points],
                y=strategy["actual_path"][:initial_n_points],
                mode="lines",
                line=dict(color=strategy["actual_color"], width=5, simplify=False),
                name=f"{strategy['label']} realized path",
                showlegend=False,
                customdata=[
                    cumulative_return_pct(value)
                    for value in strategy["actual_path"][:initial_n_points]
                ],
                hovertemplate=(
                    f"{strategy['label']} realized path<br>"
                    "Date: %{x|%Y-%m-%d}<br>"
                    "Wealth: %{y:.3f}<br>"
                    "Cumulative return: %{customdata:+.1f}%<extra></extra>"
                ),
            ),
            row=1,
            col=col,
        )
        animated_trace_indices.append(trace_index)

        # Geometric expectation.
        trace_index = len(fig.data)
        fig.add_trace(
            go.Scatter(
                x=dates[:initial_n_points],
                y=strategy["geometric_path"][:initial_n_points],
                mode="lines",
                line=dict(
                    color=expected_path_color,
                    width=2.5,
                    dash="dash",
                    simplify=False,
                ),
                name=f"{strategy['label']} geometric path",
                showlegend=False,
                hovertemplate=(
                    "Geometric path<br>"
                    "Date: %{x|%Y-%m-%d}<br>"
                    "Wealth: %{y:.3f}<extra></extra>"
                ),
            ),
            row=1,
            col=col,
        )
        animated_trace_indices.append(trace_index)

        fig.add_hline(
            y=INITIAL_VALUE,
            line=dict(color=baseline_color, width=1, dash="dot"),
            opacity=0.80,
            row=1,
            col=col,
        )

    # Bottom panel traces.
    bottom_retained_index = len(fig.data)
    fig.add_trace(
        make_retained_bar(retained_values, bar_labels, retained_return_color),
        row=2,
        col=1,
    )

    bottom_drag_index = len(fig.data)
    fig.add_trace(
        make_drag_bar(drag_values, retained_values, bar_labels, volatility_drag_color),
        row=2,
        col=1,
    )

    # Animation frames. Keep bottom bars in every frame so they persist.
    frames = []
    slider_steps = []

    frame_indices = list(range(initial_end, N_STEPS + 1, FRAME_STRIDE))
    if frame_indices[-1] != N_STEPS:
        frame_indices.append(N_STEPS)

    frame_trace_indices = animated_trace_indices + [
        bottom_retained_index,
        bottom_drag_index,
    ]

    for i in frame_indices:
        n_points = i + 1

        frame_slow_actual_return = cumulative_return_pct(slow["actual_path"][i])
        frame_chase_actual_return = cumulative_return_pct(chase["actual_path"][i])

        # Update subplot titles for this frame
        frame_subplot_titles = (
            f"Conservative Compounding<br><span style='font-size:0.92em;color:{slow['actual_color']}'>{frame_slow_actual_return:+.1f}%</span>",
            f"High Return Chase with Extreme Volatility<br><span style='font-size:0.92em;color:{chase['actual_color']}'>{frame_chase_actual_return:+.1f}%</span>",
            "Arithmetic Return and Volatility Drag",
        )

        frame_data = []

        for strategy in (slow, chase):
            for path_index in range(N_PATHS):
                frame_data.append(
                    go.Scatter(
                        x=dates[:n_points],
                        y=strategy["paths"][:n_points, path_index],
                        mode="lines",
                        line=dict(color=cloud_color, width=1.2, simplify=False),
                        showlegend=False,
                    )
                )

            frame_data.append(
                go.Scatter(
                    x=dates[:n_points],
                    y=strategy["actual_path"][:n_points],
                    mode="lines",
                    line=dict(color=strategy["actual_color"], width=5, simplify=False),
                    customdata=[
                        cumulative_return_pct(value)
                        for value in strategy["actual_path"][:n_points]
                    ],
                    showlegend=False,
                )
            )

            frame_data.append(
                go.Scatter(
                    x=dates[:n_points],
                    y=strategy["geometric_path"][:n_points],
                    mode="lines",
                    line=dict(
                        color=expected_path_color,
                        width=2.5,
                        dash="dash",
                        simplify=False,
                    ),
                    showlegend=False,
                )
            )

        frame_data.append(
            make_retained_bar(retained_values, bar_labels, retained_return_color)
        )
        frame_data.append(
            make_drag_bar(drag_values, retained_values, bar_labels, volatility_drag_color)
        )

        # Provide subplot titles for this *frame* by setting annotations
        annotation_updates = []
        for idx, annotation in enumerate(fig.layout.annotations):
            if idx == 0:
                annotation_updates.append(
                    dict(
                        x=annotation['x'],
                        y=annotation['y'],
                        xref=annotation['xref'],
                        yref=annotation['yref'],
                        text=frame_subplot_titles[0],
                        font=dict(color=off_white, size=17),
                        showarrow=False,
                    )
                )
            elif idx == 1:
                annotation_updates.append(
                    dict(
                        x=annotation['x'],
                        y=annotation['y'],
                        xref=annotation['xref'],
                        yref=annotation['yref'],
                        text=frame_subplot_titles[1],
                        font=dict(color=off_white, size=17),
                        showarrow=False,
                    )
                )
            elif idx == 2:
                annotation_updates.append(
                    dict(
                        x=annotation['x'],
                        y=annotation['y'],
                        xref=annotation['xref'],
                        yref=annotation['yref'],
                        text=frame_subplot_titles[2],
                        font=dict(color=off_white, size=17),
                        showarrow=False,
                    )
                )
            else:
                annotation_updates.append(annotation)

        frames.append(
            go.Frame(
                data=frame_data,
                traces=frame_trace_indices,
                name=f"f{i}",
                layout=dict(annotations=annotation_updates)
            )
        )

        slider_steps.append({
            "args": [
                [f"f{i}"],
                {
                    "frame": {"duration": 0, "redraw": True},
                    "mode": "immediate",
                    "transition": {"duration": 0},
                    "fromcurrent": True,
                },
            ],
            "label": f"{i / STEPS_PER_YEAR:.1f}Y",
            "method": "animate",
        })

    fig.frames = frames

    # Set initial subplot titles (first frame, t=0, so returns show +0.0%)
    # But once the animation runs or you click on slider, you'll see dynamically updated returns.

    fig.update_layout(
        title=dict(
            text="Over 15 Years, Extreme Volatility Can Destroy the Big Trade",
            x=0.5,
            font=dict(color=off_white, size=24),
        ),
        template="plotly_dark",
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        height=800,
        width=1200,
        margin=dict(t=120, b=125, r=65, l=95),
        barmode="overlay",
        bargap=0.20,
        hovermode="closest",
        showlegend=False,
        updatemenus=[{
            "type": "buttons",
            "buttons": [
                {
                    "label": "▶ Play",
                    "method": "animate",
                    "args": [
                        None,
                        {
                            "frame": {
                                "duration": FRAME_DURATION,
                                "redraw": True,
                            },
                            "transition": {"duration": 0},
                            "fromcurrent": True,
                        },
                    ],
                },
                {
                    "label": "⏸ Pause",
                    "method": "animate",
                    "args": [
                        [None],
                        {
                            "frame": {"duration": 0, "redraw": True},
                            "mode": "immediate",
                            "transition": {"duration": 0},
                            "fromcurrent": True,
                        },
                    ],
                },
            ],
            "direction": "left",
            "pad": {"r": 10, "t": 72},
            "showactive": False,
            "x": 0.10,
            "xanchor": "right",
            "y": -0.03,
            "yanchor": "top",
        }],
        sliders=[{
            "active": 0,
            "yanchor": "top",
            "xanchor": "left",
            "currentvalue": {"visible": False},
            "transition": {"duration": 0},
            "pad": {"b": 10, "t": 42},
            "len": 0.84,
            "x": 0.15,
            "y": -0.03,
            "steps": slider_steps,
        }],
    )

    # The initial annotations were set by make_subplots, but we now want to force them to match *t=0* returns
    annotation_updates = []
    t0_slow_actual_return = cumulative_return_pct(slow["actual_path"][0])
    t0_chase_actual_return = cumulative_return_pct(chase["actual_path"][0])
    init_titles = (
        f"Conservative Compounding<br><span style='font-size:0.92em;color:{slow['actual_color']}'>{t0_slow_actual_return:+.1f}%</span>",
        f"High Return Chase with Extreme Volatility<br><span style='font-size:0.92em;color:{chase['actual_color']}'>{t0_chase_actual_return:+.1f}%</span>",
        "Arithmetic Return and Volatility Drag",
    )
    for i, annotation in enumerate(fig.layout.annotations):
        if i == 0:
            annotation_updates.append(
                dict(
                    x=annotation['x'],
                    y=annotation['y'],
                    xref=annotation['xref'],
                    yref=annotation['yref'],
                    text=init_titles[0],
                    font=dict(color=off_white, size=17),
                    showarrow=False,
                )
            )
        elif i == 1:
            annotation_updates.append(
                dict(
                    x=annotation['x'],
                    y=annotation['y'],
                    xref=annotation['xref'],
                    yref=annotation['yref'],
                    text=init_titles[1],
                    font=dict(color=off_white, size=17),
                    showarrow=False,
                )
            )
        elif i == 2:
            annotation_updates.append(
                dict(
                    x=annotation['x'],
                    y=annotation['y'],
                    xref=annotation['xref'],
                    yref=annotation['yref'],
                    text=init_titles[2],
                    font=dict(color=off_white, size=17),
                    showarrow=False,
                )
            )
        else:
            annotation_updates.append(annotation)
    fig.update_layout(annotations=annotation_updates)

    for col in (1, 2):
        fig.update_xaxes(
            axis_style,
            row=1,
            col=col,
            range=[dates[0], dates[-1]],
            title_text="Date",
        )
        fig.update_yaxes(
            axis_style,
            row=1,
            col=col,
            range=[0.0, 6.0],
            title_text="Wealth multiple" if col == 1 else "",
        )

    fig.update_xaxes(
        axis_style,
        row=2,
        col=1,
        range=[-10, return_x_max],
        ticksuffix="%",
        title_text="Annual arithmetic return",
    )
    fig.update_yaxes(
        axis_style,
        row=2,
        col=1,
        showgrid=False,
        title_text="",
        categoryorder="array",
        categoryarray=bar_labels,
        autorange="reversed",
    )

    return fig

if __name__ == "__main__":
    fig = build_figure()
    fig.show()

###### ______________________________________________________________________________________________________________________________________

##### ⛰️ An Interesting Dichotomy

Maximizing Geometric Returns is a dual optimization problem...

**Maximization Problem:**

Find the portfolio (or asset allocation) that maximizes $g$:
$$
\max_{\mathbf{w}} \left[ \mu(\mathbf{w}) - \frac{1}{2} \sigma^2(\mathbf{w}) \right]
$$
where $\mathbf{w}$ is the vector of portfolio weights.

**Interpretation:**
- Increasing expected return $\mu$ increases $g$.
- But higher volatility $\sigma$ reduces $g$ quadratically ("volatility drag").

In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy.optimize import minimize


# ============================================================
# Config
# ============================================================

SEED = 17
rng = np.random.default_rng(SEED)

N_ASSETS = 7
N_RANDOM_PORTFOLIOS = 18_000
N_FRONTIER_POINTS = 120

# Plotly-native slider values. These are percentages, not decimals.
MEAN_OPTIONS_PCT = np.arange(4.0, 17.0, 1.0)
VOLATILITY_OPTIONS_PCT = np.arange(6.0, 31.0, 2.0)

DEFAULT_MEAN_PCT = 8.0
DEFAULT_VOLATILITY_PCT = 16.0

SHOW_FIG = True


# ============================================================
# Styling — transparent Plotly-dark house style
# ============================================================

TRANSPARENT = "rgba(0,0,0,0)"

OFF_WHITE = "#e0e0e0"
MUTED_WHITE = "#b8b8b8"

FRONTIER_COLOR = "#ffd84d"
CURRENT_COLOR = "#f4b200"
MIN_VARIANCE_COLOR = "#00ff88"
GUIDE_COLOR = "rgba(244,178,0,0.42)"
ISO_CAGR_COLOR = "rgba(255,216,77,0.30)"

AXIS_STYLE = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.10)",
    tickfont=dict(color=OFF_WHITE),
    linecolor=OFF_WHITE,
    zeroline=False,
    title_font=dict(color=OFF_WHITE),
)


# ============================================================
# Synthetic asset universe
# ============================================================

# The chart is intentionally self-contained. Replace these values with your
# own expected-return vector and covariance matrix when using real portfolios.
ASSET_NAMES = np.array([
    "Defensive Equity",
    "Quality Equity",
    "Broad Equity",
    "Small Cap",
    "Credit",
    "Treasuries",
    "Alternatives",
])

EXPECTED_RETURNS = np.array([
    0.060,
    0.080,
    0.095,
    0.125,
    0.070,
    0.045,
    0.105,
])

TARGET_ASSET_VOLS = np.array([
    0.125,
    0.145,
    0.175,
    0.245,
    0.105,
    0.075,
    0.185,
])

# A positive-definite two-factor covariance structure.
FACTOR_LOADINGS = np.array([
    [0.72,  0.08],
    [0.82,  0.10],
    [0.95,  0.14],
    [1.12,  0.28],
    [0.38,  0.42],
    [-0.18, 0.74],
    [0.54, -0.34],
])

FACTOR_COVARIANCE = np.array([
    [0.0225, 0.0020],
    [0.0020, 0.0100],
])

IDIOSYNCRATIC_VARIANCE = np.diag([
    0.0045,
    0.0040,
    0.0048,
    0.0095,
    0.0032,
    0.0018,
    0.0080,
])

RAW_COVARIANCE = (
    FACTOR_LOADINGS
    @ FACTOR_COVARIANCE
    @ FACTOR_LOADINGS.T
    + IDIOSYNCRATIC_VARIANCE
)

# Convert the raw factor covariance into a correlation matrix, then rescale
# to the intended annualized asset volatilities.
raw_sd = np.sqrt(np.diag(RAW_COVARIANCE))
correlation = RAW_COVARIANCE / np.outer(raw_sd, raw_sd)
COVARIANCE = (
    correlation
    * np.outer(TARGET_ASSET_VOLS, TARGET_ASSET_VOLS)
)


# ============================================================
# Portfolio mathematics
# ============================================================

def portfolio_return(weights):
    return float(weights @ EXPECTED_RETURNS)


def portfolio_variance(weights):
    return float(weights @ COVARIANCE @ weights)


def portfolio_volatility(weights):
    return np.sqrt(portfolio_variance(weights))


def approximate_geometric_return(arithmetic_return, volatility):
    """
    Continuous-time approximation:

        geometric return ≈ arithmetic return - 0.5 * volatility²
    """
    return arithmetic_return - 0.5 * volatility**2


def solve_minimum_variance(target_return=None, initial_weights=None):
    """Solve the long-only minimum-variance portfolio."""
    if initial_weights is None:
        initial_weights = np.full(N_ASSETS, 1.0 / N_ASSETS)

    constraints = [
        {
            "type": "eq",
            "fun": lambda weights: np.sum(weights) - 1.0,
        }
    ]

    if target_return is not None:
        constraints.append(
            {
                "type": "eq",
                "fun": (
                    lambda weights, target=target_return:
                    portfolio_return(weights) - target
                ),
            }
        )

    result = minimize(
        fun=portfolio_variance,
        x0=initial_weights,
        method="SLSQP",
        bounds=[(0.0, 1.0)] * N_ASSETS,
        constraints=constraints,
        options={
            "ftol": 1e-12,
            "maxiter": 2_000,
            "disp": False,
        },
    )

    if not result.success:
        raise RuntimeError(
            f"Efficient-frontier optimization failed: {result.message}"
        )

    return result.x


# ============================================================
# Random portfolios and efficient frontier
# ============================================================

# Dirichlet weights generate fully invested, long-only random portfolios.
random_weights = rng.dirichlet(
    alpha=np.full(N_ASSETS, 0.75),
    size=N_RANDOM_PORTFOLIOS,
)

random_returns = random_weights @ EXPECTED_RETURNS
random_variances = np.einsum(
    "ij,jk,ik->i",
    random_weights,
    COVARIANCE,
    random_weights,
)
random_volatilities = np.sqrt(random_variances)
random_geometric_returns = approximate_geometric_return(
    random_returns,
    random_volatilities,
)

# Global minimum-variance portfolio.
min_variance_weights = solve_minimum_variance()
min_variance_return = portfolio_return(min_variance_weights)
min_variance_volatility = portfolio_volatility(min_variance_weights)
min_variance_geometric = approximate_geometric_return(
    min_variance_return,
    min_variance_volatility,
)

# Long-only efficient frontier from the minimum-variance return to the
# highest individual asset expected return.
target_returns = np.linspace(
    min_variance_return,
    EXPECTED_RETURNS.max(),
    N_FRONTIER_POINTS,
)

frontier_weights = []
frontier_returns = []
frontier_volatilities = []

warm_start = min_variance_weights.copy()

for target_return in target_returns:
    try:
        weights = solve_minimum_variance(
            target_return=target_return,
            initial_weights=warm_start,
        )
    except RuntimeError:
        continue

    warm_start = weights
    frontier_weights.append(weights)
    frontier_returns.append(portfolio_return(weights))
    frontier_volatilities.append(portfolio_volatility(weights))

frontier_weights = np.asarray(frontier_weights)
frontier_returns = np.asarray(frontier_returns)
frontier_volatilities = np.asarray(frontier_volatilities)
frontier_geometric_returns = approximate_geometric_return(
    frontier_returns,
    frontier_volatilities,
)


# ============================================================
# Display coordinates and chart ranges
# ============================================================

random_return_pct = 100.0 * random_returns
random_volatility_pct = 100.0 * random_volatilities
random_geometric_pct = 100.0 * random_geometric_returns

frontier_return_pct = 100.0 * frontier_returns
frontier_volatility_pct = 100.0 * frontier_volatilities
frontier_geometric_pct = 100.0 * frontier_geometric_returns

min_variance_return_pct = 100.0 * min_variance_return
min_variance_volatility_pct = 100.0 * min_variance_volatility
min_variance_geometric_pct = 100.0 * min_variance_geometric

X_MIN = min(
    float(VOLATILITY_OPTIONS_PCT.min()) - 1.0,
    float(random_volatility_pct.min()) - 1.0,
)
X_MAX = max(
    float(VOLATILITY_OPTIONS_PCT.max()) + 1.0,
    float(random_volatility_pct.max()) + 1.0,
)

Y_MIN = min(
    float(MEAN_OPTIONS_PCT.min()) - 1.0,
    float(random_return_pct.min()) - 1.0,
)
Y_MAX = max(
    float(MEAN_OPTIONS_PCT.max()) + 1.0,
    float(random_return_pct.max()) + 1.0,
)


# ============================================================
# Figure
# ============================================================

fig = go.Figure()

# ------------------------------------------------------------
# Iso-geometric-return curves
# ------------------------------------------------------------

iso_x = np.linspace(X_MIN, X_MAX, 320)
iso_cagr_levels_pct = [2.0, 4.0, 6.0, 8.0, 10.0, 12.0]

for level_pct in iso_cagr_levels_pct:
    # Arithmetic mean = geometric return + 0.5 * sigma².
    # Inputs and outputs are converted consistently into percentage points.
    iso_y = level_pct + iso_x**2 / 200.0

    visible = (iso_y >= Y_MIN) & (iso_y <= Y_MAX)
    if not np.any(visible):
        continue

    x_values = iso_x[visible]
    y_values = iso_y[visible]

    label_index = int(0.72 * (len(x_values) - 1))
    labels = [""] * len(x_values)
    labels[label_index] = f"{level_pct:.0f}% geometric"

    fig.add_trace(
        go.Scatter(
            x=x_values,
            y=y_values,
            mode="lines+text",
            line=dict(
                color=ISO_CAGR_COLOR,
                width=1.2,
                dash="dot",
            ),
            text=labels,
            textposition="top center",
            textfont=dict(
                color="rgba(255,216,77,0.70)",
                size=10,
            ),
            hoverinfo="skip",
            showlegend=False,
        )
    )

# ------------------------------------------------------------
# Random portfolio cloud
# ------------------------------------------------------------

fig.add_trace(
    go.Scattergl(
        x=random_volatility_pct,
        y=random_return_pct,
        mode="markers",
        marker=dict(
            size=5,
            opacity=0.42,
            color=random_geometric_pct,
            colorscale=[
                [0.00, "#ff3030"],
                [0.50, "#ffd84d"],
                [1.00, "#18d618"],
            ],
            cmin=float(np.percentile(random_geometric_pct, 2)),
            cmax=float(np.percentile(random_geometric_pct, 98)),
            colorbar=dict(
                title=dict(
                    text="Approx.<br>geometric<br>return",
                    font=dict(color=OFF_WHITE),
                ),
                tickfont=dict(color=OFF_WHITE),
                ticksuffix="%",
                outlinecolor="rgba(255,255,255,0.25)",
                outlinewidth=1,
                thickness=15,
                len=0.72,
                x=1.015,
            ),
            line=dict(width=0),
        ),
        customdata=random_geometric_pct,
        showlegend=False,
        hovertemplate=(
            "Random portfolio<br>"
            "Annualized volatility: %{x:.2f}%<br>"
            "Arithmetic mean: %{y:.2f}%<br>"
            "Approx. geometric return: %{customdata:.2f}%"
            "<extra></extra>"
        ),
    )
)

# ------------------------------------------------------------
# Efficient frontier
# ------------------------------------------------------------

frontier_label_text = [""] * len(frontier_return_pct)
if len(frontier_label_text) > 0:
    frontier_label_text[-1] = "  Efficient frontier"

fig.add_trace(
    go.Scatter(
        x=frontier_volatility_pct,
        y=frontier_return_pct,
        mode="lines+text",
        line=dict(
            color=FRONTIER_COLOR,
            width=4,
        ),
        text=frontier_label_text,
        textposition="middle right",
        textfont=dict(
            color=FRONTIER_COLOR,
            size=12,
        ),
        cliponaxis=False,
        customdata=frontier_geometric_pct,
        showlegend=False,
        hovertemplate=(
            "Efficient frontier<br>"
            "Annualized volatility: %{x:.2f}%<br>"
            "Arithmetic mean: %{y:.2f}%<br>"
            "Approx. geometric return: %{customdata:.2f}%"
            "<extra></extra>"
        ),
    )
)

# ------------------------------------------------------------
# Global minimum-variance portfolio
# ------------------------------------------------------------

fig.add_trace(
    go.Scatter(
        x=[min_variance_volatility_pct],
        y=[min_variance_return_pct],
        mode="markers+text",
        marker=dict(
            size=13,
            color=MIN_VARIANCE_COLOR,
            symbol="diamond",
            line=dict(
                color=OFF_WHITE,
                width=1.2,
            ),
        ),
        text=["  Minimum variance"],
        textposition="bottom right",
        textfont=dict(
            color=MIN_VARIANCE_COLOR,
            size=11,
        ),
        cliponaxis=False,
        showlegend=False,
        hovertemplate=(
            "Minimum-variance portfolio<br>"
            f"Annualized volatility: {min_variance_volatility_pct:.2f}%<br>"
            f"Arithmetic mean: {min_variance_return_pct:.2f}%<br>"
            f"Approx. geometric return: {min_variance_geometric_pct:.2f}%"
            "<extra></extra>"
        ),
    )
)

# ------------------------------------------------------------
# Current-portfolio crosshairs
# ------------------------------------------------------------

horizontal_guide_index = len(fig.data)
fig.add_trace(
    go.Scatter(
        x=[X_MIN, X_MAX],
        y=[DEFAULT_MEAN_PCT, DEFAULT_MEAN_PCT],
        mode="lines",
        line=dict(
            color=GUIDE_COLOR,
            width=1.5,
            dash="dash",
        ),
        hoverinfo="skip",
        showlegend=False,
    )
)

vertical_guide_index = len(fig.data)
fig.add_trace(
    go.Scatter(
        x=[DEFAULT_VOLATILITY_PCT, DEFAULT_VOLATILITY_PCT],
        y=[Y_MIN, Y_MAX],
        mode="lines",
        line=dict(
            color=GUIDE_COLOR,
            width=1.5,
            dash="dash",
        ),
        hoverinfo="skip",
        showlegend=False,
    )
)

# ------------------------------------------------------------
# Current portfolio
# ------------------------------------------------------------

current_portfolio_index = len(fig.data)
fig.add_trace(
    go.Scatter(
        x=[DEFAULT_VOLATILITY_PCT],
        y=[DEFAULT_MEAN_PCT],
        mode="markers+text",
        marker=dict(
            size=19,
            color=CURRENT_COLOR,
            symbol="star",
            line=dict(
                color=OFF_WHITE,
                width=1.8,
            ),
        ),
        text=["  Current Portfolio"],
        textposition="top right",
        textfont=dict(
            color=OFF_WHITE,
            size=13,
        ),
        cliponaxis=False,
        showlegend=False,
        hovertemplate=(
            "Current Portfolio<br>"
            "Annualized volatility: %{x:.2f}%<br>"
            "Arithmetic mean: %{y:.2f}%<br>"
            "Read approximate CAGR from the nearest geometric-return contour"
            "<extra></extra>"
        ),
    )
)


# ============================================================
# Plotly-native sliders
# ============================================================

mean_slider_steps = []

for mean_pct in MEAN_OPTIONS_PCT:
    mean_slider_steps.append(
        {
            "label": f"{mean_pct:.0f}%",
            "method": "restyle",
            "args": [
                {
                    "y": [
                        [mean_pct],
                        [mean_pct, mean_pct],
                    ]
                },
                [
                    current_portfolio_index,
                    horizontal_guide_index,
                ],
            ],
        }
    )

volatility_slider_steps = []

for volatility_pct in VOLATILITY_OPTIONS_PCT:
    variance_pct_squared = volatility_pct**2 / 100.0

    volatility_slider_steps.append(
        {
            "label": f"{volatility_pct:.0f}%",
            "method": "restyle",
            "args": [
                {
                    "x": [
                        [volatility_pct],
                        [volatility_pct, volatility_pct],
                    ]
                },
                [
                    current_portfolio_index,
                    vertical_guide_index,
                ],
            ],
        }
    )

mean_active = int(
    np.where(np.isclose(MEAN_OPTIONS_PCT, DEFAULT_MEAN_PCT))[0][0]
)
volatility_active = int(
    np.where(
        np.isclose(VOLATILITY_OPTIONS_PCT, DEFAULT_VOLATILITY_PCT)
    )[0][0]
)


# ============================================================
# Layout
# ============================================================

fig.update_layout(
    title=dict(
        text=(
            "Efficient frontier and geometric-return optimization"
            "<br><sup>"
            "Move Current Portfolio upward by increasing arithmetic mean, "
            "or leftward by reducing volatility and variance. "
            "Approx. geometric return = arithmetic mean − ½σ²."
            "</sup>"
        ),
        x=0.5,
        font=dict(color=OFF_WHITE),
    ),
    template="plotly_dark",
    paper_bgcolor=TRANSPARENT,
    plot_bgcolor=TRANSPARENT,
    font=dict(color=OFF_WHITE),
    width=1200,
    height=800,
    margin=dict(
        t=120,
        b=170,
        r=150,
        l=90,
    ),
    hovermode="closest",
    showlegend=False,
    sliders=[
        {
            "active": mean_active,
            "x": 0.05,
            "y": -0.14,
            "len": 0.42,
            "xanchor": "left",
            "yanchor": "top",
            "transition": {"duration": 0},
            "currentvalue": {
                "visible": True,
                "prefix": "Current arithmetic mean: ",
                "font": {
                    "color": OFF_WHITE,
                    "size": 13,
                },
                "xanchor": "center",
            },
            "pad": {
                "t": 38,
                "b": 0,
            },
            "bgcolor": TRANSPARENT,
            "bordercolor": "rgba(255,255,255,0.22)",
            "borderwidth": 1,
            "tickcolor": OFF_WHITE,
            "font": {"color": OFF_WHITE},
            "steps": mean_slider_steps,
        },
        {
            "active": volatility_active,
            "x": 0.53,
            "y": -0.14,
            "len": 0.42,
            "xanchor": "left",
            "yanchor": "top",
            "transition": {"duration": 0},
            "currentvalue": {
                "visible": True,
                "prefix": "Current volatility σ: ",
                "suffix": "",
                "font": {
                    "color": OFF_WHITE,
                    "size": 13,
                },
                "xanchor": "center",
            },
            "pad": {
                "t": 38,
                "b": 0,
            },
            "bgcolor": TRANSPARENT,
            "bordercolor": "rgba(255,255,255,0.22)",
            "borderwidth": 1,
            "tickcolor": OFF_WHITE,
            "font": {"color": OFF_WHITE},
            "steps": volatility_slider_steps,
        },
    ],
)

fig.update_xaxes(
    AXIS_STYLE,
    range=[X_MIN, X_MAX],
    ticksuffix="%",
    title_text="Annualized volatility σ  —  lower variance ←",
)

fig.update_yaxes(
    AXIS_STYLE,
    range=[Y_MIN, Y_MAX],
    ticksuffix="%",
    title_text="Arithmetic mean return  —  higher mean ↑",
)

# A concise visual reminder of the preferred direction of travel.
fig.add_annotation(
    x=0.02,
    y=1.01,
    xref="paper",
    yref="paper",
    text=(
        "<b>Geometric-return improvement:</b> move toward the upper-left "
        "without crossing beyond the feasible frontier."
    ),
    showarrow=False,
    xanchor="left",
    yanchor="bottom",
    font=dict(
        color=MUTED_WHITE,
        size=12,
    ),
)


# ============================================================
# Display only — no HTML export
# ============================================================

if __name__ == "__main__" and SHOW_FIG:
    fig.show()

---

#### 💭 Closing Thoughts and Future Topics

 **📑 TL;DW Executive Summary** 
  - Volatility drag is a consequence of geometric compounding: wealth multiplies as $P_n = P_0 \prod (1+r_i)$, so long-run growth is governed by mean *log* returns. For small $r$, $\log(1+r)\approx r-\tfrac12 r^2$, hence $\mathbb{E}[\log(1+r)]\approx\mathbb{E}[r]-\tfrac12\mathrm{Var}(r)$—**mean compounded return ≈ arithmetic mean return minus volatility drag**
  - Under GBM the gap is exact: expected wealth grows at $\mu$ while typical (median / geometric-mean) wealth grows at $\mu-\tfrac12\sigma^2$. High arithmetic return with excessive volatility can leave almost nothing for compound growth (e.g. $\mu=18\%$ retaining only ~$2\%$ geometrically), while a lower-$\mu$/lower-$\sigma$ book often compounds more wealth over long horizons
  - The only thing worse than volatility is *needing liquidity into it*. Large downside moves force undesirable states—selling into drawdowns or missing the recovery. Hedging that softens gap-downs (and lets you monetize protection) reduces realized volatility drag and preserves compound growth through crashes
  - Confusingly, there is *good* and *bad* volatility drag. Temporary high volatility can be constructive after a large positive payoff, when buying the fire sale, or inside a volatility-diversified portfolio—but there is no free lunch
  - Maximizing long-run wealth is a dual problem: $\max_{\mathbf{w}}\bigl[\mu(\mathbf{w})-\tfrac12\sigma^2(\mathbf{w})\bigr]$. Raising $\mu$ helps $g$; raising $\sigma$ hurts it *quadratically*. Optimize for compound growth and survival, not eye-catching arithmetic averages

###### ______________________________________________________________________________________________________________________________________

 
**Future Topics**

Technical Videos and Other Discussions

 - Fama-French / Carhart and Factor Modeling in General
 - Hawkes Processes
 - Merton Jump Diffusion Model (and Characteristic Function Pricing, Carr-Madan 1999)
 - Market-Making Models and Simulation (Stoikov-Avellaneda)
 - My First Year as a Quant
 - Why Hedge Funds are Actually Secretive
 - Non-Markovian Models (fractional Brownian motion, Volterra Process)
 - Top 3 Uses of Linear Algebra for Quant Finance
 - Girsanov's Change of Measure
 - Rough Path Theory, Applications of Path Signatures
 - Sig-Vol Model, Calibration, and Pricing
 - Trading with Alternative Data Sources
 - Pairs Trading and Statistical Arbitrage
 - Data Cleaning & Outlier Handling in Financial Time Series
 - Practical Issues in Multi-Asset Portfolio Backtesting
 - Risk Premia Harvesting: Equity, FX, Rates

[Ideas for Interactive Brokers Apps and Tutorials](https://www.interactivebrokers.com/mkt/?src=quantguildY&url=%2Fen%2Fwhyib%2Foverview.php)

- How Interactive Broker's API Works (EWrapper/EClient)
- How to Backtest a Trading Strategy with Interactive Brokers
- Algorithmic Volatility Trading System


---

####  $\text{Copyright © 2026 Quant Guild} \quad \quad \quad \quad \text{Author: Roman Paolucci}$